# 1. Granger Causality

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import grangercausalitytests
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

# Load the dataset
try:
    df = pd.read_csv('DC_Master_ARIMA_Filled_pct_change_dummies.csv', parse_dates=['Date'], index_col='Date')
    df = df[df.index <= '2024-10-01']
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: 'DC_Master_Levels_and_Stationary.csv' not found.")
    # Exit or handle error appropriately if file not found
    exit()
except Exception as e:
    print(f"Error loading dataset: {e}")
    # Exit or handle other loading errors
    exit()

stationary_cols = [
    'Unemployment_Rate', # Already I(0)
    'Interest_Rate_pct_change',
    'Mortgage_Rate_pct_change',
    'CPI_pct_change',
    'House_Index_pct_change',
    'Poverty_Rate_pct_change',
    'Median_Household_Income_pct_change',
    'GDP_pct_change',
    'Population_pct_change'
]

# Verify columns exist in the DataFrame
missing_cols = [col for col in stationary_cols if col not in df.columns]
if missing_cols:
    print(f"Error: The following expected stationary columns are missing: {missing_cols}")
    # Filter out missing columns to proceed or exit
    stationary_cols = [col for col in stationary_cols if col in df.columns]
    if not stationary_cols:
        print("No stationary columns found to perform tests. Exiting.")
        exit()
    else:
        print(f"Proceeding with available columns: {stationary_cols}")


# Select only the stationary columns
df_stationary = df[stationary_cols].copy()

# --- Handle Missing Values ---
# Granger causality tests require non-missing values.
# Differencing creates NaNs at the beginning. Forecasts might have NaNs at the end.
initial_rows = len(df_stationary)
df_stationary.dropna(inplace=True)
final_rows = len(df_stationary)
print(f"Removed {initial_rows - final_rows} rows with NaN values.")

if final_rows < 10: # Need sufficient data points for the test
     print(f"Warning: Only {final_rows} complete observations available after handling NaNs. Results might be unreliable.")
     if final_rows == 0:
         print("Error: No complete observations left after removing NaNs. Cannot perform Granger Causality tests.")
         exit()


# --- Perform Granger Causality Tests ---
maxlag = 8
test_results = {}
variables = df_stationary.columns

print(f"\nPerforming Granger Causality tests for maxlag={maxlag}...")

for cause_var in variables:
    for effect_var in variables:
        if cause_var == effect_var:
            continue # Skip testing variable on itself

        print(f"Testing: {cause_var} -> {effect_var}")
        try:
            # Ensure data is float64 for the test
            test_data = df_stationary[[effect_var, cause_var]].astype(np.float64)
            # Check for constant columns which cause errors in the test
            if test_data[cause_var].nunique() <= 1 or test_data[effect_var].nunique() <= 1:
                 print(f"Skipping {cause_var} -> {effect_var} due to constant data.")
                 test_results[(cause_var, effect_var)] = {'error': 'Constant data detected'}
                 continue

            gc_result = grangercausalitytests(test_data, maxlag=maxlag, verbose=False)
            test_results[(cause_var, effect_var)] = gc_result

        except Exception as e:
            print(f"Error testing {cause_var} -> {effect_var}: {e}")
            test_results[(cause_var, effect_var)] = {'error': str(e)}


# --- Present Results ---
print("\n--- Granger Causality Test Results (p-values for F-test) ---")
print(f"Significance level: 0.1 (p < 0.1 suggests Granger causality)")

results_summary = []
for (cause_var, effect_var), result_dict in test_results.items():
    if 'error' in result_dict:
        results_summary.append({
            'Cause': cause_var,
            'Effect': effect_var,
            'Lag': 'N/A',
            'P-Value': 'Error',
            'Significant (p<0.1)': 'N/A',
            'Details': result_dict['error']
        })
        continue

    min_p_value = 1.0
    significant_lags = []
    for lag in range(1, maxlag + 1):
         # Accessing the F-test p-value: result_dict[lag][0]['ssr_ftest'][1]
         try:
            p_value = result_dict[lag][0]['ssr_ftest'][1]
            min_p_value = min(min_p_value, p_value)
            if p_value < 0.1:
                significant_lags.append(lag)
         except (IndexError, KeyError, TypeError):
             # Handle cases where test might not return expected structure for a lag
             print(f"Warning: Could not retrieve p-value for lag {lag} in {cause_var} -> {effect_var}")
             continue


    results_summary.append({
        'Cause': cause_var,
        'Effect': effect_var,
        'Min P-Value': f"{min_p_value:.4f}" if min_p_value <= 1.0 else 'N/A',
        'Significant Lags (p<0.1)': ', '.join(map(str, significant_lags)) if significant_lags else 'None'
    })

# Convert summary to DataFrame for better display
results_df = pd.DataFrame(results_summary)

# Display the summary table
print("\nSummary of Granger Causality Tests:")
print(results_df.to_string())

# Optional: Display detailed results for significant relationships
print("\n--- Detailed p-values for relationships with at least one significant lag ---")
significant_found = False
for (cause_var, effect_var), result_dict in test_results.items():
    if 'error' in result_dict: continue

    is_significant = False
    p_values_str = []
    for lag in range(1, maxlag + 1):
         try:
            p_value = result_dict[lag][0]['ssr_ftest'][1]
            p_str = f"{p_value:.3f}"
            if p_value < 0.1:
                p_str += "*"
                is_significant = True
            p_values_str.append(f"Lag {lag}: {p_str}")
         except (IndexError, KeyError, TypeError):
             p_values_str.append(f"Lag {lag}: N/A")


    if is_significant:
        significant_found = True
        print(f"\n{cause_var} -> {effect_var}:")
        print(" | ".join(p_values_str))

if not significant_found:
    print("No significant Granger causality found at p < 0.1 for any lag up to", maxlag)

# 2. Feature Engineering

In [ ]:
import pandas as pd
import numpy as np # Import numpy for potential future use, though not strictly needed here

# Load the dataset
try:
    # Make sure the original CSV has the 'House_Index' column and all source variables
    df = pd.read_csv("DC_Master_ARIMA_Filled_pct_change_dummies.csv")
    print("Source CSV loaded successfully.")
except FileNotFoundError:
    print("Error: The file 'DC_Master_ARIMA_Filled_pct_change_dummies.csv' was not found.")
    # Exit or raise error if the file is essential
    raise FileNotFoundError("Input CSV not found.")
except Exception as e:
    print(f"An error occurred loading the CSV: {e}")
    raise

# --- Feature Engineering ---

# Check if Date column exists and process it
date_col = None
if 'Date' in df.columns:
    date_col = 'Date'
    try:
        # Convert to datetime
        df[date_col] = pd.to_datetime(df[date_col])
        # Set Date as index
        df = df.set_index(date_col)
        print("Processed 'Date' column and set as index.")
    except Exception as e:
        print(f"Error processing Date column: {e}. Proceeding without datetime index.")
        # Reset df index if setting failed partially
        if date_col in df.columns: # Check if it was added but not set as index
             df = df.reset_index(drop=True)
        date_col = None # Ensure date_col is None if processing failed
else:
    print("Warning: 'Date' column not found. Proceeding without a datetime index.")


# Identify columns
target_col = 'House_Index_pct_change'
original_target_col = 'House_Index' # Column with original levels to keep

# Identify exogenous pct_change columns (excluding the target pct_change)
pct_change_cols = [col for col in df.columns if col.endswith('_pct_change') and col != target_col]
# Identify other specific exogenous columns
unemployment_col = 'Unemployment_Rate' # Assuming this is the raw rate, not pct_change
# Identify dummy columns
dummy_cols = [col for col in df.columns if col.startswith('dummy_')] # More robust way to find dummies

# Define which columns need lags 1-8 generated
# Start with the target pct_change
cols_to_lag = [target_col]
# Add exogenous pct_change columns
cols_to_lag.extend(pct_change_cols)
# Add Unemployment_Rate if it exists
if unemployment_col in df.columns:
    cols_to_lag.append(unemployment_col)
else:
    print(f"Warning: Specified unemployment column '{unemployment_col}' not found in source data.")

print(f"\nTarget column (pct_change): {target_col}")
print(f"Original value column to keep: {original_target_col}")
print(f"Exogenous pct_change columns identified: {pct_change_cols}")
print(f"Other exogenous columns identified: {[unemployment_col] if unemployment_col in df.columns else 'None'}")
print(f"Columns identified for lagging (1-8): {cols_to_lag}")
print(f"Dummy columns identified to keep: {dummy_cols}")

# Initialize the engineered DataFrame using the index from df
df_eng = pd.DataFrame(index=df.index)

# --- Keep Original Target Level ---
if original_target_col in df.columns:
    df_eng[original_target_col] = df[original_target_col]
    print(f"Kept original target level column: '{original_target_col}'")
else:
    # This is critical, raise an error if missing
    raise ValueError(f"Original target column '{original_target_col}' not found in the input dataframe. Cannot proceed.")

# --- Keep Original Target Percentage Change ---
if target_col in df.columns:
    df_eng[target_col] = df[target_col]
    print(f"Kept target percentage change column: '{target_col}'")
else:
    # This is also critical
    raise ValueError(f"Target percentage change column '{target_col}' not found in the input dataframe. Cannot proceed.")

# --- *** CORRECTED SECTION START *** ---
# --- Keep Original Exogenous Variables (Non-Lagged) ---
# These are the variables for which we also create lags
exog_cols_to_keep_original = pct_change_cols + ([unemployment_col] if unemployment_col in df.columns else [])

print("\nKeeping original (non-lagged) exogenous variables...")
for col in exog_cols_to_keep_original:
    if col in df.columns:
        df_eng[col] = df[col]
        print(f"  - Kept original column: {col}")
    else:
        # This case should ideally not happen if identification logic is correct, but good to have a warning
        print(f"  - Warning: Original exogenous column '{col}' was identified but not found in source df.")
# --- *** CORRECTED SECTION END *** ---

# --- Create Lagged Features ---
print("\nCreating lagged features...")
num_lags = 8
for col in cols_to_lag:
    if col in df.columns:
        for i in range(1, num_lags + 1):
            df_eng[f'{col}_lag{i}'] = df[col].shift(i)
        print(f"  - Created lags 1-{num_lags} for: {col}")
    else:
        # This warning indicates an issue with the cols_to_lag list generation
        print(f"  - Warning: Column '{col}' specified for lagging not found in source df.")

# --- Add Time Trend ---
df_eng['time_trend'] = range(len(df_eng))
print("Added 'time_trend' feature.")

# --- Add Dummy Variables ---
print("\nAdding dummy variables...")
kept_dummies = []
for col in dummy_cols:
    if col in df.columns:
        df_eng[col] = df[col]
        kept_dummies.append(col)
    else:
        print(f"  - Warning: Identified dummy column '{col}' not found in source df.")
if kept_dummies:
    print(f"  - Kept dummy columns: {kept_dummies}")
else:
    print("  - No dummy variables found or kept.")


# --- Save the DataFrame Before Dropping NaNs ---
# This file contains the full feature set including originals and lags, but with NaNs from shifting
output_filename_missing = 'PreSelect_Missing.csv' # New name to avoid confusion
try:
    df_eng.to_csv(output_filename_missing, index=True)
    print(f"\nFeature engineered DataFrame saved to '{output_filename_missing}' (includes NaNs).")
    print("\nDataFrame Info (before dropping NaNs):")
    df_eng.info()
    print("\nFirst few rows (includes NaNs):")
    print(df_eng.head(num_lags + 2))
except Exception as e:
    print(f"Error saving file '{output_filename_missing}': {e}")

# --- Drop NaNs and Filter Date Range for Final Modeling File ---
df_eng_final = df_eng.copy()
initial_rows = len(df_eng_final)
df_eng_final.dropna(inplace=True)
rows_after_na = len(df_eng_final)
print(f"\nDataFrame rows before dropping NaNs: {initial_rows}")
print(f"DataFrame rows after dropping NaNs: {rows_after_na} ({initial_rows - rows_after_na} rows removed)")

# Apply date filter if the index is datetime
rows_before_date_filter = len(df_eng_final)
filter_date = '2024-10-01' # Use the same filter date as before
date_filtered = False
if pd.api.types.is_datetime64_any_dtype(df_eng_final.index):
    try:
        filter_date_dt = pd.to_datetime(filter_date)
        df_eng_final = df_eng_final[df_eng_final.index <= filter_date_dt]
        print(f"Applied date filter: index <= '{filter_date}'")
        date_filtered = True
    except Exception as e:
         print(f"Error applying date filter to index: {e}")
else:
    print(f"Could not apply date filter: Index is not datetime type.")

if date_filtered:
    rows_after_date_filter = len(df_eng_final)
    print(f"DataFrame rows before date filter: {rows_before_date_filter}")
    print(f"DataFrame rows after date filter: {rows_after_date_filter} ({rows_before_date_filter - rows_after_date_filter} rows removed)")


# Save the cleaned and filtered DataFrame (ready for modeling)
output_filename_final = 'Preselect.csv' # New name
try:
    df_eng_final.to_csv(output_filename_final, index=True)
    print(f"\nFeature engineered DataFrame (cleaned, filtered) saved to '{output_filename_final}'.")
    print("\nFinal DataFrame Info:")
    df_eng_final.info()
except Exception as e:
    print(f"Error saving file '{output_filename_final}': {e}")

print("\n--- Feature Engineering Script Finished ---")


# 3. ElasticNet/Lasso

### 3.1 Walk-Forward Validation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV, ElasticNetCV, Lasso, ElasticNet
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
import warnings
import time # Added for timing operations

# Ignore common warnings from sklearn internal operations
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning) # Add deprecation warnings if needed

# --- Configuration ---
# Input file containing selected features, target pct_change, and original index level
file_path = 'Preselect.csv'
# Target variable (percentage change - for prediction)
target_pct_change_col = 'House_Index_pct_change'
# Original level column (for evaluation and iterative prediction)
target_level_col = 'House_Index'
# Date column name
date_col = 'Date'
# Define the training and testing periods
# Assuming the CSV is sorted by date
# Let's define the split date (adjust as needed)
train_end_date = pd.to_datetime('2015-10-01')  # Convert to datetime
test_start_date = pd.to_datetime('2016-01-01')  # Convert to datetime

# Walk-Forward Validation Setup
n_cv_splits = 9 # Number of splits for TimeSeriesSplit during hyperparameter tuning
max_horizon = 8 # Maximum forecast steps ahead (e.g., predict 1 to 8 steps)
cv_lasso_alphas = np.logspace(-6, 1, 100) # Alphas for LassoCV
cv_elasticnet_alphas = np.logspace(-6, 1, 100) # Alphas for ElasticNetCV
cv_l1_ratios = np.arange(0.01, 1.01, 0.01) # L1 ratios for ElasticNetCV

# --- Load Data ---
print(f"Loading data from '{file_path}'...")
try:
    df = pd.read_csv(file_path)
    print(f"Columns in loaded data: {df.columns.tolist()}") # Debug print
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
    raise

# Convert Date column to datetime objects and set as index
if date_col in df.columns:
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col)
    df = df.sort_index() # Ensure data is sorted by date
    print("Data loaded, 'Date' column set as index, and sorted successfully.")
else:
     print(f"Error: '{date_col}' column not found. Cannot proceed without a date index.")
     raise ValueError(f"'{date_col}' column not found in {file_path}")

# --- Prepare Data ---
# Drop rows with NaN in target or features (essential after lagging)
df.dropna(subset=[target_pct_change_col, target_level_col], inplace=True)
feature_cols = [col for col in df.columns if col not in [target_pct_change_col, target_level_col]]
# Ensure no NaNs remain in feature columns after potential lagging/differencing
df.dropna(subset=feature_cols, inplace=True)

X = df[feature_cols]
y_pct_change = df[target_pct_change_col]
y_level = df[target_level_col]

# Identify target-related lagged features for iterative updates
# Assumes lagged features contain the original target name, e.g., 'House_Index_pct_change_lag1'
house_index_features = [col for col in feature_cols if target_pct_change_col in col and '_lag' in col]
exogenous_features = [col for col in feature_cols if col not in house_index_features]

# Extract lag numbers from target features
lag_numbers = {}
for feature in house_index_features:
    parts = feature.split('_lag')
    if len(parts) > 1:
        try:
            lag_numbers[feature] = int(parts[1])
        except ValueError:
            print(f"Warning: Could not extract lag number from {feature}")
            continue
print(f"Identified {len(house_index_features)} lagged target features.")
print(f"Identified {len(exogenous_features)} exogenous features.")
print(f"Lag numbers found: {lag_numbers}")

# --- Split Data: Initial Training and Full Test Set ---
X_train_initial = X[X.index <= pd.to_datetime(train_end_date)]
y_train_pct_change_initial = y_pct_change[y_pct_change.index <= pd.to_datetime(train_end_date)]
y_train_level_initial = y_level[y_level.index <= pd.to_datetime(train_end_date)]

X_test = X[X.index >= pd.to_datetime(test_start_date)]
y_test_pct_change = y_pct_change[y_pct_change.index >= pd.to_datetime(test_start_date)]
y_test_level = y_level[y_level.index >= pd.to_datetime(test_start_date)]

if X_test.empty:
    raise ValueError("Test set is empty. Adjust train_end_date or test_start_date.")

print(f"Initial training data shape: {X_train_initial.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Initial training period: {X_train_initial.index.min()} to {X_train_initial.index.max()}")
print(f"Test period: {X_test.index.min()} to {X_test.index.max()}")

# --- Walk-Forward Validation with Iterative Tuning ---

# Store predictions for each horizon
# Dictionary where key is horizon (1 to max_horizon), value is another dict {'lasso': [], 'elasticnet': [], 'actual': [], 'last_known_level': []}
all_predictions = {h: {'lasso': [], 'elasticnet': [], 'actual': [], 'last_known_level': []} for h in range(1, max_horizon + 1)}
test_indices = X_test.index

start_time_wf = time.time()

# Outer loop: Iterate through each point in the test set to start predictions from
# We stop early enough to have actuals for the max_horizon prediction
for i in range(len(X_test) - max_horizon + 1):
    current_test_date = test_indices[i]
    wf_step_start_time = time.time()

    print(f"\nWalk-Forward Step {i+1}/{len(X_test) - max_horizon + 1}: Predicting from {current_test_date.strftime('%Y-%m-%d')}...")

    # --- 1. Define Current Training Window (Expanding Window) ---
    # Includes all initial training data plus test data up to the point *before* the current test point
    current_train_end_date = test_indices[i-1] if i > 0 else train_end_date
    X_train_current = X[X.index <= current_train_end_date]
    y_train_pct_change_current = y_pct_change[y_pct_change.index <= current_train_end_date]
    y_train_level_current = y_level[y_level.index <= current_train_end_date]

    # --- 2. Scale Data (Refit on Current Training Window) ---
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_current)
    # Scale the single row of features we will use to start the multi-step prediction
    # Ensure X_test.iloc[[i]] is a DataFrame to get feature names
    X_test_step_scaled = scaler.transform(X_test.iloc[[i]]) # This is a numpy array

    # Convert scaled training data back to DataFrame for CV input (keeps index if needed, though CV uses position)
    X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train_current.index, columns=X_train_current.columns)

    # Store scaler's means and stds for inverse transforming or manual scaling later
    feature_means = dict(zip(feature_cols, scaler.mean_))
    feature_stds = dict(zip(feature_cols, scaler.scale_))
    # Handle zero std dev (if a feature is constant in the training window)
    for k, v in feature_stds.items():
        if v == 0:
            print(f"Warning: Feature '{k}' has zero standard deviation in current training window. Replacing std with 1.")
            feature_stds[k] = 1.0


    # --- 3. Iterative Hyperparameter Tuning (on Current Training Window) ---
    print("   Tuning hyperparameters...")
    tuning_start_time = time.time()
    tscv = TimeSeriesSplit(n_splits=n_cv_splits)

    # LassoCV
    lasso_cv = LassoCV(alphas=cv_lasso_alphas, cv=tscv, n_jobs=-1, random_state=42, max_iter=10000)
    lasso_cv.fit(X_train_scaled, y_train_pct_change_current)
    best_lasso_alpha = lasso_cv.alpha_

    # ElasticNetCV
    elasticnet_cv = ElasticNetCV(alphas=cv_elasticnet_alphas, l1_ratio=cv_l1_ratios, cv=tscv, n_jobs=-1, random_state=42, max_iter=10000)
    elasticnet_cv.fit(X_train_scaled, y_train_pct_change_current)
    best_elasticnet_alpha = elasticnet_cv.alpha_
    best_elasticnet_l1_ratio = elasticnet_cv.l1_ratio_

    print(f"   Tuning complete ({time.time() - tuning_start_time:.2f}s). Best Lasso alpha: {best_lasso_alpha:.6f}, Best ElasticNet alpha: {best_elasticnet_alpha:.6f}, l1_ratio: {best_elasticnet_l1_ratio:.2f}")

    # --- 4. Train Final Models (on Current Training Window with Best Hyperparameters) ---
    print("   Training final models...")
    lasso_model = Lasso(alpha=best_lasso_alpha, random_state=42, max_iter=10000)
    elasticnet_model = ElasticNet(alpha=best_elasticnet_alpha, l1_ratio=best_elasticnet_l1_ratio, random_state=42, max_iter=10000)

    lasso_model.fit(X_train_scaled, y_train_pct_change_current)
    elasticnet_model.fit(X_train_scaled, y_train_pct_change_current)

    # --- 5. Iterative Multi-Step Prediction ---
    print(f"   Performing {max_horizon}-step iterative prediction...")
    # Use the last known actual level from the *current training set* to start the level prediction
    last_known_actual_level = y_train_level_current.iloc[-1]

    # Initialize feature sets for iterative updates (start with the scaled features for the current test step)
    # Convert the numpy array X_test_step_scaled back to a DataFrame for easier feature manipulation by name
    current_lasso_features_scaled_df = pd.DataFrame(X_test_step_scaled, index=[current_test_date], columns=feature_cols)
    current_elasticnet_features_scaled_df = pd.DataFrame(X_test_step_scaled, index=[current_test_date], columns=feature_cols)

    # Track the predicted levels step-by-step
    lasso_last_pred_level = last_known_actual_level
    elasticnet_last_pred_level = last_known_actual_level

    # Store the sequence of pct_change predictions made within this multi-step forecast
    # Needed to update lagged features correctly
    lasso_pct_preds_sequence = []
    elasticnet_pct_preds_sequence = []

    for step in range(1, max_horizon + 1):
        # Predict 1 step ahead based on *current* features
        # Models expect numpy array
        lasso_pred_pct = lasso_model.predict(current_lasso_features_scaled_df.values)[0]
        elasticnet_pred_pct = elasticnet_model.predict(current_elasticnet_features_scaled_df.values)[0]

        lasso_pct_preds_sequence.append(lasso_pred_pct)
        elasticnet_pct_preds_sequence.append(elasticnet_pred_pct)

        # Calculate predicted level for this step
        lasso_pred_level = lasso_last_pred_level * (1 + lasso_pred_pct)
        elasticnet_pred_level = elasticnet_last_pred_level * (1 + elasticnet_pred_pct)

        # Store the prediction for this horizon
        actual_level_for_step = y_test_level.iloc[i + step - 1]
        all_predictions[step]['lasso'].append(lasso_pred_level)
        all_predictions[step]['elasticnet'].append(elasticnet_pred_level)
        all_predictions[step]['actual'].append(actual_level_for_step)
        # Store the level from *before* this n-step prediction started
        all_predictions[step]['last_known_level'].append(last_known_actual_level)


        # --- Feature Update for the NEXT step's prediction (if not the last step) ---
        if step < max_horizon:
            # Create copies of the current feature DataFrames to update
            lasso_next_features_unscaled = {}
            elasticnet_next_features_unscaled = {}

            # Unscale the current features for easier manipulation
            for col in feature_cols:
                 lasso_next_features_unscaled[col] = current_lasso_features_scaled_df[col].values[0] * feature_stds[col] + feature_means[col]
                 elasticnet_next_features_unscaled[col] = current_elasticnet_features_scaled_df[col].values[0] * feature_stds[col] + feature_means[col]


            # Update lagged target features
            for feature, lag in lag_numbers.items():
                if lag == 1:
                    # Lag 1 uses the prediction we just made for the current step
                    lasso_next_features_unscaled[feature] = lasso_pred_pct
                    elasticnet_next_features_unscaled[feature] = elasticnet_pred_pct
                else:
                    # Lag > 1 uses the value from the feature with lag-1 *from the previous step's features*
                    prev_lag_feature = feature.replace(f"_lag{lag}", f"_lag{lag-1}")
                    if prev_lag_feature in feature_cols:
                        # We need the *unscaled* value of the lag-1 feature from the *previous* time step
                        lasso_next_features_unscaled[feature] = lasso_next_features_unscaled[prev_lag_feature] # Value shifts 'down'
                        elasticnet_next_features_unscaled[feature] = elasticnet_next_features_unscaled[prev_lag_feature] # Value shifts 'down'
                    else:
                         print(f"Warning: Could not find feature {prev_lag_feature} to update {feature}")


            # Update exogenous features (carry forward the last known value - from the original X_test.iloc[[i]])
            # No explicit update needed here as we started with X_test.iloc[[i]] and only update target lags

            # Scale the updated features *using the same scaler fitted on X_train_current*
            lasso_next_features_scaled = {}
            elasticnet_next_features_scaled = {}
            for col in feature_cols:
                 lasso_next_features_scaled[col] = (lasso_next_features_unscaled[col] - feature_means[col]) / feature_stds[col]
                 elasticnet_next_features_scaled[col] = (elasticnet_next_features_unscaled[col] - feature_means[col]) / feature_stds[col]


            # Update the feature DataFrames for the next iteration of the inner loop
            # Create a new index for the next predicted time step (not strictly necessary but good practice)
            next_pred_time = current_test_date + pd.Timedelta(days=step) # Simplistic, assumes daily/regular data
            current_lasso_features_scaled_df = pd.DataFrame([lasso_next_features_scaled], index=[next_pred_time], columns=feature_cols)
            current_elasticnet_features_scaled_df = pd.DataFrame([elasticnet_next_features_scaled], index=[next_pred_time], columns=feature_cols)

            # Update the 'last predicted level' for the next step's calculation
            lasso_last_pred_level = lasso_pred_level
            elasticnet_last_pred_level = elasticnet_pred_level

    # End of inner loop (multi-step prediction)
    print(f"   Walk-forward step {i+1} completed in {time.time() - wf_step_start_time:.2f}s")

# End of outer loop (walk-forward validation)
total_wf_time = time.time() - start_time_wf
print(f"\nWalk-Forward Validation complete. Total time: {total_wf_time:.2f} seconds ({total_wf_time/60:.2f} minutes).")

# --- Calculate Metrics for Each Horizon ---
results_summary = []

print("\n--- Calculating Metrics for Each Forecast Horizon ---")

for h in range(1, max_horizon + 1):
    print(f"\nHorizon: {h}-Step Ahead")
    preds_lasso = np.array(all_predictions[h]['lasso'])
    preds_elasticnet = np.array(all_predictions[h]['elasticnet'])
    actuals = np.array(all_predictions[h]['actual'])
    last_knowns = np.array(all_predictions[h]['last_known_level'])

    if len(actuals) == 0:
        print("   No predictions generated for this horizon.")
        continue

    # --- Standard Metrics ---
    # Handle potential division by zero in MAPE
    mape_actuals = np.where(actuals == 0, 1e-9, actuals) # Replace 0s with small number

    mse_lasso = mean_squared_error(actuals, preds_lasso)
    rmse_lasso = np.sqrt(mse_lasso)
    mae_lasso = mean_absolute_error(actuals, preds_lasso)
    mape_lasso = mean_absolute_percentage_error(mape_actuals, preds_lasso) * 100

    mse_elasticnet = mean_squared_error(actuals, preds_elasticnet)
    rmse_elasticnet = np.sqrt(mse_elasticnet)
    mae_elasticnet = mean_absolute_error(actuals, preds_elasticnet)
    mape_elasticnet = mean_absolute_percentage_error(mape_actuals, preds_elasticnet) * 100

    # --- Directional Accuracy ---
    actual_direction = np.sign(actuals - last_knowns)
    lasso_pred_direction = np.sign(preds_lasso - last_knowns)
    elasticnet_pred_direction = np.sign(preds_elasticnet - last_knowns)

    # Handle cases where prediction == last_known (sign is 0) - count as incorrect? Or neutral?
    # Common practice: count 0 as incorrect unless actual is also 0. Let's count matches.
    dir_acc_lasso = np.mean(actual_direction == lasso_pred_direction) * 100
    dir_acc_elasticnet = np.mean(actual_direction == elasticnet_pred_direction) * 100

    print("  Lasso:")
    print(f"    MSE: {mse_lasso:.4f}, RMSE: {rmse_lasso:.4f}, MAE: {mae_lasso:.4f}, MAPE: {mape_lasso:.2f}%, Dir. Acc.: {dir_acc_lasso:.2f}%")
    print("  ElasticNet:")
    print(f"    MSE: {mse_elasticnet:.4f}, RMSE: {rmse_elasticnet:.4f}, MAE: {mae_elasticnet:.4f}, MAPE: {mape_elasticnet:.2f}%, Dir. Acc.: {dir_acc_elasticnet:.2f}%")

    results_summary.append({
        'Horizon': h,
        'Lasso_MSE': mse_lasso, 'Lasso_RMSE': rmse_lasso, 'Lasso_MAE': mae_lasso, 'Lasso_MAPE': mape_lasso, 'Lasso_Dir_Acc': dir_acc_lasso,
        'ElasticNet_MSE': mse_elasticnet, 'ElasticNet_RMSE': rmse_elasticnet, 'ElasticNet_MAE': mae_elasticnet, 'ElasticNet_MAPE': mape_elasticnet, 'ElasticNet_Dir_Acc': dir_acc_elasticnet,
    })

# --- Create Summary DataFrames ---
results_df = pd.DataFrame(results_summary)
results_df.set_index('Horizon', inplace=True)

# Separate DFs for cleaner presentation
lasso_results_df = results_df[[col for col in results_df.columns if 'Lasso' in col]].copy()
lasso_results_df.columns = [col.replace('Lasso_', '') for col in lasso_results_df.columns]

elasticnet_results_df = results_df[[col for col in results_df.columns if 'ElasticNet' in col]].copy()
elasticnet_results_df.columns = [col.replace('ElasticNet_', '') for col in elasticnet_results_df.columns]

print("\n--- Detailed Results by Forecast Horizon ---")
print("\nLasso Model Results:")
print(lasso_results_df.round(4))

print("\nElasticNet Model Results:")
print(elasticnet_results_df.round(4))

# --- Average Performance ---
avg_metrics = results_df.mean()

avg_comparison = pd.DataFrame({
    'Model': ['Lasso', 'ElasticNet'],
    'Avg MSE': [avg_metrics['Lasso_MSE'], avg_metrics['ElasticNet_MSE']],
    'Avg RMSE': [avg_metrics['Lasso_RMSE'], avg_metrics['ElasticNet_RMSE']],
    'Avg MAE': [avg_metrics['Lasso_MAE'], avg_metrics['ElasticNet_MAE']],
    'Avg MAPE (%)': [avg_metrics['Lasso_MAPE'], avg_metrics['ElasticNet_MAPE']],
    'Avg Dir. Acc. (%)': [avg_metrics['Lasso_Dir_Acc'], avg_metrics['ElasticNet_Dir_Acc']]
})

print(f"\n--- Average Performance Across All Forecast Horizons (1-{max_horizon} Steps) ---")
print(avg_comparison.round({'Avg MSE': 4, 'Avg RMSE': 4, 'Avg MAE': 4, 'Avg MAPE (%)': 2, 'Avg Dir. Acc. (%)': 2}))

# --- Comparison: 1-Step vs Average ---
step1_comparison = pd.DataFrame({
    'Model': ['Lasso 1-Step', 'Lasso Avg (1-8)', 'ElasticNet 1-Step', 'ElasticNet Avg (1-8)'],
    'MSE': [results_df.loc[1, 'Lasso_MSE'], avg_metrics['Lasso_MSE'], results_df.loc[1, 'ElasticNet_MSE'], avg_metrics['ElasticNet_MSE']],
    'RMSE': [results_df.loc[1, 'Lasso_RMSE'], avg_metrics['Lasso_RMSE'], results_df.loc[1, 'ElasticNet_RMSE'], avg_metrics['ElasticNet_RMSE']],
    'MAE': [results_df.loc[1, 'Lasso_MAE'], avg_metrics['Lasso_MAE'], results_df.loc[1, 'ElasticNet_MAE'], avg_metrics['ElasticNet_MAE']],
    'MAPE (%)': [results_df.loc[1, 'Lasso_MAPE'], avg_metrics['Lasso_MAPE'], results_df.loc[1, 'ElasticNet_MAPE'], avg_metrics['ElasticNet_MAPE']],
    'Dir. Acc. (%)': [results_df.loc[1, 'Lasso_Dir_Acc'], avg_metrics['Lasso_Dir_Acc'], results_df.loc[1, 'ElasticNet_Dir_Acc'], avg_metrics['ElasticNet_Dir_Acc']]
})

print("\n--- Comparison: 1-Step vs Average of All Steps ---")
print(step1_comparison.round({'MSE': 4, 'RMSE': 4, 'MAE': 4, 'MAPE (%)': 2, 'Dir. Acc. (%)': 2}))

print("\nAnalysis complete.")



### 3.2 Visualization of Walk-Forward Validation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
import warnings

# Ignore potential warnings from matplotlib
warnings.filterwarnings('ignore', category=UserWarning)

print("Creating visualizations for forecast comparisons...")

# --- Assumptions ---
# This script assumes the following variables exist in the environment
# from the execution of the 'iterative_tuning_forecasting_v2' script:
# - y_train_level_initial: Pandas Series with the initial training target levels (index=Date)
# - y_test_level: Pandas Series with the test target levels (index=Date)
# - all_predictions: Dictionary storing predictions for each horizon
#   (e.g., all_predictions[h]['lasso'], all_predictions[h]['elasticnet'], all_predictions[h]['actual'])
# - max_horizon: The maximum forecast horizon used (integer)

# --- Check if variables exist (optional but good practice) ---
required_vars = ['y_train_level_initial', 'y_test_level', 'all_predictions', 'max_horizon']
if not all(var in globals() for var in required_vars):
    raise NameError("One or more required variables (y_train_level_initial, y_test_level, all_predictions, max_horizon) not found. Please run the forecasting script first.")

# --- Configuration ---
horizons_to_plot = [1, 4, 8] # Select horizons to visualize (ensure they are <= max_horizon)
# Filter horizons to only those available
horizons_to_plot = [h for h in horizons_to_plot if h <= max_horizon and h in all_predictions]

if not horizons_to_plot:
     raise ValueError(f"None of the specified horizons {horizons_to_plot} are available in the results (max_horizon={max_horizon}).")

print(f"Plotting horizons: {horizons_to_plot}")

# --- Prepare Data for Plotting ---

# Training data
train_dates = y_train_level_initial.index
train_values = y_train_level_initial.values

# Test data (actual values)
test_dates = y_test_level.index
test_values = y_test_level.values

# Function to extract aligned dates and forecast values for a given horizon
def get_aligned_forecast_data(horizon):
    """
    Extracts predictions and their corresponding dates for a specific horizon.

    Args:
        horizon (int): The forecast horizon (e.g., 1, 4, 8).

    Returns:
        dict: Dictionary containing 'dates', 'lasso_values',
              'elasticnet_values', 'actual_values'.
              Returns None if data for the horizon is missing or empty.
    """
    if horizon not in all_predictions or not all_predictions[horizon]['actual']:
        print(f"Warning: No data found for horizon {horizon}.")
        return None

    preds_lasso = np.array(all_predictions[horizon]['lasso'])
    preds_elasticnet = np.array(all_predictions[horizon]['elasticnet'])
    actuals = np.array(all_predictions[horizon]['actual'])
    num_preds = len(actuals)

    # The k-th prediction for horizon 'h' corresponds to the actual value
    # at index k + h - 1 in the original y_test_level Series.
    # (Using 0-based indexing for iloc)
    start_idx = horizon - 1
    end_idx = start_idx + num_preds
    forecast_dates = y_test_level.index[start_idx:end_idx]

    # Basic check
    if len(forecast_dates) != num_preds:
         print(f"Warning: Date alignment mismatch for horizon {horizon}. Expected {num_preds} dates, got {len(forecast_dates)}.")
         # Attempt correction or return None based on desired robustness
         # For now, let's assume the length matches if indices are valid
         if len(forecast_dates) < num_preds:
              print("   Adjusting number of predictions to match available dates.")
              num_preds = len(forecast_dates)
              preds_lasso = preds_lasso[:num_preds]
              preds_elasticnet = preds_elasticnet[:num_preds]
              actuals = actuals[:num_preds]
         elif len(forecast_dates) > num_preds:
             print("   Adjusting number of dates to match available predictions.")
             forecast_dates = forecast_dates[:num_preds]


    return {
        'dates': forecast_dates,
        'lasso_values': preds_lasso,
        'elasticnet_values': preds_elasticnet,
        'actual_values': actuals # Actual values corresponding to the forecast dates
    }

# Extract data for selected horizons
forecast_data = {}
for h in horizons_to_plot:
    data = get_aligned_forecast_data(h)
    if data:
        forecast_data[h] = data

# Determine plot limits
# plot_start_date = train_dates.min() # Original start date
display_start_date = pd.to_datetime('2015-01-01') # New start date set to 2015
plot_end_date = test_dates.max() # Keep end date as the max test date

# --- Create Plots ---
num_plots = len(forecast_data)
if num_plots == 0:
    print("No forecast data available to plot.")
else:
    fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True) # Use shared x-axis

    # Define colors for horizons
    colors = plt.cm.viridis(np.linspace(0, 0.8, len(horizons_to_plot))) # Example colormap

    # --- Plot for Lasso ---
    ax1 = axes[0]
    ax1.plot(train_dates, train_values, 'k-', label='Training Data', linewidth=1.5, alpha=0.7)
    ax1.plot(test_dates, test_values, color='gray', linestyle='--', label='Actual Test Data', linewidth=1.5, alpha=0.7)

    for i, h in enumerate(horizons_to_plot):
        if h in forecast_data:
            ax1.plot(forecast_data[h]['dates'], forecast_data[h]['lasso_values'],
                     color=colors[i], linestyle='-', label=f'{h}-Step Forecast', linewidth=1.2)

    ax1.set_title('Lasso Model: Actual vs Forecasted House Index', fontsize=14)
    ax1.set_ylabel('House Index', fontsize=12)
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='best')
    ax1.set_xlim(display_start_date, plot_end_date) # Use new start date

    # --- Plot for ElasticNet ---
    ax2 = axes[1]
    ax2.plot(train_dates, train_values, 'k-', label='Training Data', linewidth=1.5, alpha=0.7)
    ax2.plot(test_dates, test_values, color='gray', linestyle='--', label='Actual Test Data', linewidth=1.5, alpha=0.7)

    for i, h in enumerate(horizons_to_plot):
         if h in forecast_data:
            ax2.plot(forecast_data[h]['dates'], forecast_data[h]['elasticnet_values'],
                     color=colors[i], linestyle='-', label=f'{h}-Step Forecast', linewidth=1.2)

    ax2.set_title('ElasticNet Model: Actual vs Forecasted House Index', fontsize=14)
    ax2.set_xlabel('Date', fontsize=12)
    ax2.set_ylabel('House Index', fontsize=12)
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc='best')
    ax2.set_xlim(display_start_date, plot_end_date) # Use new start date

    # Format x-axis dates
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax2.xaxis.set_major_locator(mdates.YearLocator()) # Major ticks per year
    ax2.xaxis.set_minor_locator(mdates.MonthLocator(interval=3)) # Minor ticks every 3 months
    plt.xticks(rotation=45, ha='right')

    plt.tight_layout() # Adjust layout to prevent overlap

    # --- Save and Show Plot ---
    try:
        plt.savefig('house_index_forecast_comparison.png', dpi=300, bbox_inches='tight')
        print("Plot saved as 'house_index_forecast_comparison.png'")
    except Exception as e:
        print(f"Error saving plot: {e}")

    plt.show()


### 3.3 ElasticNet/Lasso Forecast

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV, ElasticNetCV, Lasso, ElasticNet
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
import time # Added for timing operations

# Ignore common warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

print("--- Starting Final Model Training and Forecasting ---")

# --- Configuration ---
file_path = 'Preselect.csv'
target_pct_change_col = 'House_Index_pct_change'
target_level_col = 'House_Index'
date_col = 'Date'

# Date from which to start forecasting (exclusive)
# Model will be trained on all data UP TO AND INCLUDING this date.
final_train_end_date_str = '2024-10-01'
num_forecast_steps = 8

# CV parameters for final tuning (reuse from previous script if desired)
n_cv_splits = 9 # Or use the value found previously
cv_lasso_alphas = np.logspace(-6, 1, 100)
cv_elasticnet_alphas = np.logspace(-6, 1, 100)
cv_l1_ratios = [0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0]

# --- 1. Reload and Prepare Data ---
print(f"Loading data from '{file_path}'...")
try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
    raise

# Convert Date column, set index, sort
if date_col in df.columns:
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col)
    df = df.sort_index()
    print("Data loaded and processed.")
else:
     raise ValueError(f"'{date_col}' column not found in {file_path}")

# Drop NaNs from essential columns (target, features used)
# Need feature columns list again
all_cols = df.columns.tolist()
feature_cols = [col for col in all_cols if col not in [target_pct_change_col, target_level_col]]
df.dropna(subset=[target_pct_change_col, target_level_col], inplace=True)
df.dropna(subset=feature_cols, inplace=True) # Drop rows if any feature has NaN

X = df[feature_cols]
y_pct_change = df[target_pct_change_col]
y_level = df[target_level_col]

# Re-identify lagged features for iterative updates
house_index_features = [col for col in feature_cols if target_pct_change_col in col and '_lag' in col]
lag_numbers = {}
for feature in house_index_features:
    parts = feature.split('_lag')
    if len(parts) > 1:
        try:
            lag_numbers[feature] = int(parts[1])
        except ValueError:
            continue
print(f"Identified {len(house_index_features)} lagged target features.")

# --- 2. Define Final Training Set ---
try:
    final_train_end_date = pd.to_datetime(final_train_end_date_str)
    # Ensure the date exists or find the latest date <= it
    if final_train_end_date not in X.index:
         print(f"Warning: Exact date {final_train_end_date_str} not found in index.")
         final_train_end_date = X[X.index <= final_train_end_date].index.max()
         print(f"Using last available date: {final_train_end_date.strftime('%Y-%m-%d')} for final training.")
except Exception as e:
    print(f"Error processing final_train_end_date: {e}")
    raise

X_train_final = X[X.index <= final_train_end_date]
y_train_pct_change_final = y_pct_change[y_pct_change.index <= final_train_end_date]
y_train_level_final = y_level[y_level.index <= final_train_end_date]

if X_train_final.empty:
    raise ValueError(f"Final training set is empty. Check final_train_end_date ('{final_train_end_date_str}') and data range.")

print(f"Final training data shape: {X_train_final.shape}")
print(f"Final training period ends: {final_train_end_date.strftime('%Y-%m-%d')}")

# --- 3. Final Scaling ---
final_scaler = StandardScaler()
X_train_final_scaled = final_scaler.fit_transform(X_train_final)
print("Final scaler fitted on all training data.")
# Store means/stds for iterative forecast scaling
final_feature_means = dict(zip(feature_cols, final_scaler.mean_))
final_feature_stds = dict(zip(feature_cols, final_scaler.scale_))
# Handle potential zero std dev
for k, v in final_feature_stds.items():
    if v == 0:
        print(f"Warning: Feature '{k}' has zero std dev in final training data. Replacing std with 1.")
        final_feature_stds[k] = 1.0

# --- 4. Final Hyperparameter Tuning ---
print("Tuning hyperparameters on final training data...")
tuning_start_time = time.time()
tscv = TimeSeriesSplit(n_splits=n_cv_splits)

# LassoCV
final_lasso_cv = LassoCV(alphas=cv_lasso_alphas, cv=tscv, n_jobs=-1, random_state=42, max_iter=10000)
final_lasso_cv.fit(X_train_final_scaled, y_train_pct_change_final)
final_best_lasso_alpha = final_lasso_cv.alpha_

# ElasticNetCV
final_elasticnet_cv = ElasticNetCV(alphas=cv_elasticnet_alphas, l1_ratio=cv_l1_ratios, cv=tscv, n_jobs=-1, random_state=42, max_iter=10000)
final_elasticnet_cv.fit(X_train_final_scaled, y_train_pct_change_final)
final_best_elasticnet_alpha = final_elasticnet_cv.alpha_
final_best_elasticnet_l1_ratio = final_elasticnet_cv.l1_ratio_

print(f"Final tuning complete ({time.time() - tuning_start_time:.2f}s).")
print(f"  Best Lasso alpha: {final_best_lasso_alpha:.6f}")
print(f"  Best ElasticNet alpha: {final_best_elasticnet_alpha:.6f}, l1_ratio: {final_best_elasticnet_l1_ratio:.2f}")

# --- 5. Train Final Models ---
print("Training final models on all available data...")
final_lasso_model = Lasso(alpha=final_best_lasso_alpha, random_state=42, max_iter=10000)
final_elasticnet_model = ElasticNet(alpha=final_best_elasticnet_alpha, l1_ratio=final_best_elasticnet_l1_ratio, random_state=42, max_iter=10000)

final_lasso_model.fit(X_train_final_scaled, y_train_pct_change_final)
final_elasticnet_model.fit(X_train_final_scaled, y_train_pct_change_final)
print("Final models trained.")

# --- 6. Prepare for Forecasting ---
last_known_level = y_train_level_final.iloc[-1]
last_date = y_train_level_final.index[-1]

# Get the features corresponding to the last date
last_features_row = X.loc[[last_date]]
# Scale these features using the FINAL scaler
current_features_scaled = final_scaler.transform(last_features_row)
current_features_scaled_df = pd.DataFrame(current_features_scaled, index=[last_date], columns=feature_cols)

print(f"Forecasting starting after {last_date.strftime('%Y-%m-%d')} using level {last_known_level:.4f}")

# --- 7. Iterative Forecasting Loop ---
forecast_results = []
lasso_last_pred_level = last_known_level
elasticnet_last_pred_level = last_known_level

# Determine date frequency for generating future dates
# Try to infer frequency, default to Monthly Start ('MS') if ambiguous
inferred_freq = pd.infer_freq(X_train_final.index)
print(f"Inferred data frequency: {inferred_freq}")
if inferred_freq is None:
    print("Warning: Could not infer frequency. Assuming Monthly Start ('MS'). Adjust offset if needed.")
    date_offset = pd.tseries.offsets.DateOffset(months=1) # Adjust if daily ('D'), quarterly ('QS'), etc.
else:
    # Use inferred frequency to generate offsets
    date_offset = pd.tseries.frequencies.to_offset(inferred_freq)

current_pred_date = last_date # Initialize date for loop

for step in range(1, num_forecast_steps + 1):
    # Predict 1 step ahead based on *current* features
    lasso_pred_pct = final_lasso_model.predict(current_features_scaled_df.values)[0]
    elasticnet_pred_pct = final_elasticnet_model.predict(current_features_scaled_df.values)[0]

    # Calculate predicted level for this step
    lasso_pred_level = lasso_last_pred_level * (1 + lasso_pred_pct)
    elasticnet_pred_level = elasticnet_last_pred_level * (1 + elasticnet_pred_pct)

    # Calculate the date for this forecast step
    current_pred_date = current_pred_date + date_offset
    forecast_date_str = current_pred_date.strftime('%Y-%m-%d')

    # Store results
    forecast_results.append({
        'Date': forecast_date_str,
        'Lasso_Forecast': lasso_pred_level,
        'ElasticNet_Forecast': elasticnet_pred_level
    })

    # --- Feature Update for the NEXT step's prediction ---
    if step < num_forecast_steps:
        lasso_next_features_unscaled = {}
        elasticnet_next_features_unscaled = {}

        # Unscale the current features
        for col in feature_cols:
            lasso_next_features_unscaled[col] = current_features_scaled_df[col].values[0] * final_feature_stds[col] + final_feature_means[col]
            elasticnet_next_features_unscaled[col] = current_features_scaled_df[col].values[0] * final_feature_stds[col] + final_feature_means[col]

        # Update lagged target features
        for feature, lag in lag_numbers.items():
            if lag == 1:
                lasso_next_features_unscaled[feature] = lasso_pred_pct
                elasticnet_next_features_unscaled[feature] = elasticnet_pred_pct
            else:
                prev_lag_feature = feature.replace(f"_lag{lag}", f"_lag{lag-1}")
                if prev_lag_feature in feature_cols:
                    lasso_next_features_unscaled[feature] = lasso_next_features_unscaled[prev_lag_feature]
                    elasticnet_next_features_unscaled[feature] = elasticnet_next_features_unscaled[prev_lag_feature]
                # else: pass (feature remains unchanged)

        # Exogenous features are carried forward (no update needed here)

        # Scale the updated features using the FINAL scaler
        lasso_next_features_scaled = {}
        elasticnet_next_features_scaled = {}
        for col in feature_cols:
            lasso_next_features_scaled[col] = (lasso_next_features_unscaled[col] - final_feature_means[col]) / final_feature_stds[col]
            elasticnet_next_features_scaled[col] = (elasticnet_next_features_unscaled[col] - final_feature_means[col]) / final_feature_stds[col]

        # Update the feature DataFrame for the next iteration
        current_features_scaled_df = pd.DataFrame([lasso_next_features_scaled], index=[current_pred_date], columns=feature_cols)
        # Note: ElasticNet uses the same feature update logic based on its own predictions,
        # but since features are exogenous or based on pct_change (which we update identically here),
        # we only need one feature set updated. If models predicted different exogenous vars, this would change.

        # Update the 'last predicted level' for the next step's calculation
        lasso_last_pred_level = lasso_pred_level
        elasticnet_last_pred_level = elasticnet_pred_level
    # End feature update block
# End forecast loop

# --- 8. Print Forecasts ---
forecast_df = pd.DataFrame(forecast_results)
forecast_df['Date'] = pd.to_datetime(forecast_df['Date'])
forecast_df = forecast_df.set_index('Date')

print(f"\n--- Forecasts for the next {num_forecast_steps} steps after {last_date.strftime('%Y-%m-%d')} ---")
print(forecast_df.round(4))

# --- 9. Visualization ---
print("\nGenerating final forecast plot...")
plt.figure(figsize=(15, 7))

# Plot historical data (complete history up to forecast start)
plt.plot(y_level.index, y_level.values, 'k-', label='Historical Actual Data', linewidth=1.5)

# Combine last historical point with forecasts for smooth lines
lasso_plot_index = pd.Index([last_date]).union(forecast_df.index)
# *** EDITED LINE BELOW: Replaced .append() with pd.concat() ***
lasso_plot_values = pd.concat([pd.Series([last_known_level], index=[last_date]), forecast_df['Lasso_Forecast']])

elasticnet_plot_index = pd.Index([last_date]).union(forecast_df.index)
# *** EDITED LINE BELOW: Replaced .append() with pd.concat() ***
elasticnet_plot_values = pd.concat([pd.Series([last_known_level], index=[last_date]), forecast_df['ElasticNet_Forecast']])

# Plot forecasts
plt.plot(lasso_plot_index, lasso_plot_values, 'b-o', label='Lasso Forecast', linewidth=1.5, markersize=4)
plt.plot(elasticnet_plot_index, elasticnet_plot_values, 'g-^', label='ElasticNet Forecast', linewidth=1.5, markersize=4)

# Formatting
plt.title(f'House Index: Historical Data and {num_forecast_steps}-Step Forecast after {last_date.strftime("%Y-%m-%d")}', fontsize=14)
plt.ylabel('House Index', fontsize=12)
plt.xlabel('Date', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

# Set x-axis limits (e.g., show last few years of history + forecast)
display_history_years = 5
viz_start_date = last_date - pd.DateOffset(years=display_history_years)
viz_end_date = forecast_df.index.max() + pd.DateOffset(months=1) # Add a little padding
plt.xlim(viz_start_date, viz_end_date)

# Format date axis
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.gca().xaxis.set_minor_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45, ha='right')

plt.tight_layout()

# Save and Show Plot
try:
    plt.savefig('final_house_index_forecast.png', dpi=300, bbox_inches='tight')
    print("Plot saved as 'final_house_index_forecast.png'")
except Exception as e:
    print(f"Error saving plot: {e}")

plt.show()

print("\n--- Final forecasting process complete. ---")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

# Ignore common warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

print("--- Generating Feature Importance Plots for Final Lasso and ElasticNet Models ---")

# --- Assumptions ---
# This script assumes the following variables exist in the environment
# from the execution of the final Lasso/ElasticNet forecast script:
# - final_lasso_model: Trained Lasso model object
# - final_elasticnet_model: Trained ElasticNet model object
# - feature_cols: List of feature names in the order used for training X_train_final

# --- Check if variables exist (optional but good practice) ---
required_vars = ['final_lasso_model', 'final_elasticnet_model', 'feature_cols']
if not all(var in globals() for var in required_vars):
    raise NameError("One or more required variables (final_lasso_model, final_elasticnet_model, feature_cols) not found. Please run the final forecast script first.")

# --- Extract Coefficients ---
try:
    lasso_coef = final_lasso_model.coef_
    elasticnet_coef = final_elasticnet_model.coef_
except AttributeError:
    print("Error: Could not retrieve coefficients. Ensure models are fitted.")
    raise

# Ensure number of coefficients matches number of features
if len(lasso_coef) != len(feature_cols) or len(elasticnet_coef) != len(feature_cols):
    raise ValueError("Mismatch between number of coefficients and feature names.")

# --- Create Importance DataFrames ---
# Use absolute values for importance magnitude
lasso_importance = pd.Series(np.abs(lasso_coef), index=feature_cols).sort_values(ascending=False)
elasticnet_importance = pd.Series(np.abs(elasticnet_coef), index=feature_cols).sort_values(ascending=False)

# Filter out features with zero importance (especially relevant for Lasso)
lasso_importance_filtered = lasso_importance[lasso_importance > 1e-6] # Use a small threshold
elasticnet_importance_filtered = elasticnet_importance[elasticnet_importance > 1e-6]

if lasso_importance_filtered.empty:
    print("Warning: Lasso model selected no features (all coefficients are zero or near-zero).")
if elasticnet_importance_filtered.empty:
     print("Warning: ElasticNet model selected no features (all coefficients are zero or near-zero).")


# --- Create Plots ---
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(10, 12), sharex=True) # Share x-axis for magnitude comparison
fig.suptitle('Feature Importance (Absolute Coefficient Values)', fontsize=16)

# Plot Lasso Importances
if not lasso_importance_filtered.empty:
    ax1 = axes[0]
    # Plot horizontal bars (features on y-axis)
    lasso_importance_filtered.plot(kind='barh', ax=ax1, color='skyblue')
    ax1.set_title('Lasso Model')
    ax1.invert_yaxis() # Display most important feature at the top
    ax1.set_xlabel('Absolute Coefficient Value')
    ax1.grid(axis='x', linestyle='--', alpha=0.6)
else:
     axes[0].text(0.5, 0.5, 'Lasso selected no features', ha='center', va='center', fontsize=12)
     axes[0].set_title('Lasso Model')
     axes[0].set_yticks([])


# Plot ElasticNet Importances
if not elasticnet_importance_filtered.empty:
    ax2 = axes[1]
    elasticnet_importance_filtered.plot(kind='barh', ax=ax2, color='lightcoral')
    ax2.set_title('ElasticNet Model')
    ax2.invert_yaxis() # Display most important feature at the top
    ax2.set_xlabel('Absolute Coefficient Value')
    ax2.grid(axis='x', linestyle='--', alpha=0.6)
else:
    axes[1].text(0.5, 0.5, 'ElasticNet selected no features', ha='center', va='center', fontsize=12)
    axes[1].set_title('ElasticNet Model')
    axes[1].set_yticks([])


plt.tight_layout(rect=[0, 0.03, 1, 0.96]) # Adjust layout to prevent title overlap

# Save and Show Plot
try:
    plt.savefig('lasso_elasticnet_feature_importance.png', dpi=300, bbox_inches='tight')
    print("\nPlot saved as 'lasso_elasticnet_feature_importance.png'")
except Exception as e:
    print(f"Error saving plot: {e}")

plt.show()

print("\n--- Feature importance visualization complete. ---")


# 4. SVR

### 4.1 Single Tuning SVR Validation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
# *** EDITED: Import RandomizedSearchCV ***
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from scipy.stats import uniform, loguniform # For parameter distributions
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import warnings
import time

# Ignore common warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

print("--- Starting Univariate SVR Walk-Forward Validation (Singular Tuning with RandomizedSearch) ---") # Title updated

# --- Configuration ---
# Input file containing features and original index level
file_path = 'Preselect.csv' # Use the corrected file
# Target variable (percentage change - for prediction)
target_pct_change_col = 'House_Index_pct_change'
# Original level column (for evaluation and iterative prediction)
target_level_col = 'House_Index'
# Date column name
date_col = 'Date'

# Define the training and testing periods
train_end_date_str = '2015-12-31' # End date for initial training and tuning
test_start_date_str = '2016-01-01' # Start date for walk-forward validation
# Optional: Define an overall end date if the CSV goes further than needed
final_historical_date_str = '2024-10-01' # Last date of real historical data

# Walk-Forward Validation Setup
n_cv_splits = 9 # Number of splits for TimeSeriesSplit during initial hyperparameter tuning
max_lag = 3     # Max lag of the target variable to use as features (as per user finding)
max_horizon = 8 # Maximum forecast steps ahead

# --- *** EDITED: Define SVR Parameter Distributions for RandomizedSearch *** ---
n_tuning_iterations = 100000 # Number of parameter settings to sample (adjust as needed)

svr_param_dist = {
    # C often spans orders of magnitude, loguniform is suitable
    'C': np.arange(1, 101,0.1), # Sample C between 0.1 and 200 (log scale)
    # Epsilon is often smaller, uniform might be okay, or loguniform on smaller range
    'epsilon': uniform(loc=0.0001, scale=0.1) # Sample epsilon uniformly between 0.0001 and 0.1001
    # Alternative epsilon: loguniform(1e-4, 1e-1)
}
# --- 1. Load Data ---
print(f"Loading data from '{file_path}'...")
try:
    df = pd.read_csv(file_path, index_col=date_col, parse_dates=True)
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    df.sort_index(inplace=True)
    print("Data loaded and indexed.")
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
    raise
except Exception as e:
    print(f"Error loading {file_path}: {e}")
    raise

# --- 2. Prepare Data ---
# Filter data to a specific end date if needed
final_historical_date = pd.to_datetime(final_historical_date_str)
df_filtered = df[df.index <= final_historical_date].copy()
print(f"Data filtered up to {final_historical_date_str}. Shape: {df_filtered.shape}")
if df_filtered.empty:
    raise ValueError("DataFrame is empty after filtering by final historical date.")

# Select only necessary columns: lagged target features, target pct change, target level
lagged_target_features = []
for lag in range(1, max_lag + 1): # Use configured max_lag
    col_name = f"{target_pct_change_col}_lag{lag}"
    if col_name in df_filtered.columns:
        lagged_target_features.append(col_name)
    else:
        raise ValueError(f"Required lag feature '{col_name}' not found.")

required_cols = lagged_target_features + [target_pct_change_col, target_level_col]
missing_cols = [col for col in required_cols if col not in df_filtered.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df_selected = df_filtered[required_cols].copy()

# Handle Missing Values (should already be handled if using Preselect_Corrected.csv, but good practice)
initial_rows = len(df_selected)
df_selected.dropna(inplace=True)
rows_after_na_drop = len(df_selected)
if initial_rows > rows_after_na_drop:
    print(f"Dropped {initial_rows - rows_after_na_drop} rows containing NaNs.")
if df_selected.empty:
    raise ValueError("DataFrame empty after selecting columns and dropping NaNs.")

# Define final X (features), y (targets)
X = df_selected[lagged_target_features]
y_pct_change = df_selected[target_pct_change_col]
y_level = df_selected[target_level_col]
feature_names = X.columns.tolist()

# --- 3. Split Data ---
train_end_date = pd.to_datetime(train_end_date_str)
test_start_date = pd.to_datetime(test_start_date_str)

X_train_initial = X[X.index <= train_end_date]
y_train_pct_change_initial = y_pct_change[y_pct_change.index <= train_end_date]
y_train_level_initial = y_level[y_level.index <= train_end_date] # Levels for initial history

X_test = X[X.index >= test_start_date]
y_test_pct_change = y_pct_change[y_pct_change.index >= test_start_date]
y_test_level = y_level[y_level.index >= test_start_date] # Actual levels for evaluation

if X_test.empty or X_train_initial.empty:
    raise ValueError("Training or Test set is empty after split.")

print(f"\nInitial training data shape: {X_train_initial.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Initial training period: {X_train_initial.index.min()} to {X_train_initial.index.max()}")
print(f"Test period: {X_test.index.min()} to {X_test.index.max()}")

# --- 4. Initial Scaling (Fit ONLY on Initial Training Data) ---
print("\nScaling features (fitting only on initial training data)...")
initial_scaler = StandardScaler()
X_train_initial_scaled = initial_scaler.fit_transform(X_train_initial)
# Store the scaler's attributes for consistent use later
initial_scaler_means = initial_scaler.mean_
initial_scaler_stds = initial_scaler.scale_
# Handle potential zero std dev
initial_scaler_stds = np.where(initial_scaler_stds == 0, 1.0, initial_scaler_stds)
print("Initial scaler fitted.")

# --- 5. Hyperparameter Tuning (Performed ONCE using RandomizedSearch) ---
print(f"\nTuning Linear SVR hyperparameters ONCE using RandomizedSearch (n_iter={n_tuning_iterations}, cv={n_cv_splits})...") # Updated print
tuning_start_time = time.time()
tscv = TimeSeriesSplit(n_splits=n_cv_splits)
svr_linear = SVR(kernel='linear', max_iter=200000)

# --- *** EDITED: Use RandomizedSearchCV *** ---
random_search = RandomizedSearchCV(
    estimator=svr_linear,
    param_distributions=svr_param_dist,
    n_iter=n_tuning_iterations, # Number of parameter settings that are sampled
    cv=tscv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1,
    random_state=42 # For reproducibility
)
# Tune on the scaled initial training data
random_search.fit(X_train_initial_scaled, y_train_pct_change_initial)
# --- *** End Edit *** ---

# --- *** EDITED: Get best params from random_search object *** ---
best_params = random_search.best_params_
# Store the single set of best parameters found
fixed_best_svr_c = best_params['C']
fixed_best_svr_epsilon = best_params['epsilon']
# --- *** End Edit *** ---

print(f"Initial tuning complete ({time.time() - tuning_start_time:.2f}s).")
# Use scientific notation for potentially small epsilon for clarity
print(f"Best parameters found: C={fixed_best_svr_c:.5f}, epsilon={fixed_best_svr_epsilon:.5f}")
print("--- These fixed parameters will be used for all walk-forward steps ---")


# --- 6. Walk-Forward Validation with Fixed Hyperparameters ---
# (Rest of the script remains the same as before)

# Store predictions for each horizon
all_predictions_svr = {h: {'preds': [], 'actual': [], 'last_known_level': []} for h in range(1, max_horizon + 1)}
test_indices = X_test.index

start_time_wf = time.time()

# Initialize history using the initial training data
history_X = X_train_initial.copy()
history_y_pct_change = y_train_pct_change_initial.copy()
history_y_level = y_train_level_initial.copy()

# Outer loop: Iterate through each point in the test set to start predictions from
for i in range(len(X_test) - max_horizon + 1):
    current_test_date = test_indices[i] # Date of the first feature set used for this forecast sequence
    wf_step_start_time = time.time()

    print(f"\nWalk-Forward Step {i+1}/{len(X_test) - max_horizon + 1}: Predicting from {current_test_date.strftime('%Y-%m-%d')}...")

    # --- 6.1. Define Current Training Window Data (Raw) ---
    X_train_current = history_X
    y_train_pct_change_current = history_y_pct_change
    y_train_level_current = history_y_level

    # --- 6.2. Scale Current Training Data using Initial Scaler ---
    X_train_current_scaled = initial_scaler.transform(X_train_current)

    # --- 6.3. Train SVR Model with Fixed Hyperparameters ---
    svr_model = SVR(kernel='linear', C=fixed_best_svr_c, epsilon=fixed_best_svr_epsilon, max_iter=200000)
    svr_model.fit(X_train_current_scaled, y_train_pct_change_current)

    # --- 6.4. Iterative Multi-Step Prediction ---
    last_known_actual_level = y_train_level_current.iloc[-1]
    first_step_features_raw = X_test.iloc[[i]]
    current_features_scaled_np = initial_scaler.transform(first_step_features_raw)
    current_pred_level = last_known_actual_level

    for step in range(1, max_horizon + 1):
        svr_pred_pct = svr_model.predict(current_features_scaled_np)[0]
        current_pred_level = current_pred_level * (1 + svr_pred_pct)
        actual_level_for_step = y_test_level.iloc[i + step - 1]
        all_predictions_svr[step]['preds'].append(current_pred_level)
        all_predictions_svr[step]['actual'].append(actual_level_for_step)
        all_predictions_svr[step]['last_known_level'].append(last_known_actual_level)

        if step < max_horizon:
            current_features_unscaled = (current_features_scaled_np * initial_scaler_stds) + initial_scaler_means
            current_features_unscaled_dict = dict(zip(feature_names, current_features_unscaled[0]))
            next_features_unscaled_dict = current_features_unscaled_dict.copy()
            for lag in range(max_lag, 0, -1):
                current_lag_col = f"{target_pct_change_col}_lag{lag}"
                if lag == 1:
                    next_features_unscaled_dict[current_lag_col] = svr_pred_pct
                else:
                    prev_lag_col = f"{target_pct_change_col}_lag{lag-1}"
                    next_features_unscaled_dict[current_lag_col] = current_features_unscaled_dict[prev_lag_col]
            next_features_unscaled_np = np.array([next_features_unscaled_dict[name] for name in feature_names]).reshape(1, -1)
            current_features_scaled_np = initial_scaler.transform(next_features_unscaled_np)

    # --- 6.5 Update History ---
    history_X = pd.concat([history_X, X_test.iloc[[i]]])
    history_y_pct_change = pd.concat([history_y_pct_change, y_test_pct_change.iloc[[i]]])
    history_y_level = pd.concat([history_y_level, y_test_level.iloc[[i]]])

# End of outer loop
total_wf_time = time.time() - start_time_wf
print(f"\nSVR Walk-Forward Validation (Singular Tuning - RandomizedSearch) complete. Total time: {total_wf_time:.2f} seconds ({total_wf_time/60:.2f} minutes).") # Updated print


# --- 7. Calculate Metrics for Each Horizon ---
results_summary_svr = []
print("\n--- Calculating SVR Metrics for Each Forecast Horizon ---")

# --- Helper Functions ---
def mean_absolute_percentage_error_np(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    if np.sum(mask) == 0: return np.nan
    if len(y_true[mask]) == 0: return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def directional_accuracy_np(y_true, y_pred, y_true_prev):
    y_true, y_pred, y_true_prev = np.array(y_true), np.array(y_pred), np.array(y_true_prev)
    min_len = min(len(y_true), len(y_pred), len(y_true_prev))
    if min_len == 0: return np.nan
    y_true, y_pred, y_true_prev = y_true[:min_len], y_pred[:min_len], y_true_prev[:min_len]
    actual_diff = np.sign(y_true - y_true_prev)
    pred_diff = np.sign(y_pred - y_true_prev)
    correct_direction = (actual_diff == pred_diff).astype(int)
    return np.mean(correct_direction) * 100
# --- End Helper Functions ---

for h in range(1, max_horizon + 1):
    print(f"\nHorizon: {h}-Step Ahead (SVR - Singular Tune - RandomizedSearch)") # Updated print
    preds = np.array(all_predictions_svr[h]['preds'])
    actuals = np.array(all_predictions_svr[h]['actual'])
    last_knowns = np.array(all_predictions_svr[h]['last_known_level'])

    if len(actuals) == 0:
        print("   No predictions generated for this horizon.")
        results_summary_svr.append({'Horizon': h, 'MSE': np.nan, 'RMSE': np.nan, 'MAE': np.nan, 'MAPE': np.nan, 'Dir_Acc': np.nan})
        continue

    min_len_eval = min(len(preds), len(actuals), len(last_knowns))
    preds = preds[:min_len_eval]
    actuals = actuals[:min_len_eval]
    last_knowns = last_knowns[:min_len_eval]

    if min_len_eval == 0:
         print("   No valid aligned predictions/actuals for metric calculation.")
         results_summary_svr.append({'Horizon': h, 'MSE': np.nan, 'RMSE': np.nan, 'MAE': np.nan, 'MAPE': np.nan, 'Dir_Acc': np.nan})
         continue

    mse = mean_squared_error(actuals, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(actuals, preds)
    mape = mean_absolute_percentage_error_np(actuals, preds)
    da = directional_accuracy_np(actuals, preds, last_knowns)

    print(f"  MSE: {mse:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}, MAPE: {mape:.2f}%, Dir. Acc.: {da:.2f}%")

    results_summary_svr.append({
        'Horizon': h, 'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'Dir_Acc': da,
    })

# --- 8. Create Summary DataFrames ---
svr_results_df = pd.DataFrame(results_summary_svr)
svr_results_df.set_index('Horizon', inplace=True)

print("\n--- Detailed SVR Results by Forecast Horizon (Singular Tune - RandomizedSearch) ---") # Updated print
print(svr_results_df.round({'MSE': 4, 'RMSE': 4, 'MAE': 4, 'MAPE': 2, 'Dir_Acc': 2}))

# --- Average Performance ---
avg_metrics_svr = svr_results_df.mean()
print(f"\n--- Average SVR Performance Across All Forecast Horizons (1-{max_horizon} Steps, Singular Tune - RandomizedSearch) ---") # Updated print
print(f"  Avg MSE: {avg_metrics_svr['MSE']:.4f}")
print(f"  Avg RMSE: {avg_metrics_svr['RMSE']:.4f}")
print(f"  Avg MAE: {avg_metrics_svr['MAE']:.4f}")
print(f"  Avg MAPE (%): {avg_metrics_svr['MAPE']:.2f}")
print(f"  Avg Dir. Acc. (%): {avg_metrics_svr['Dir_Acc']:.2f}")

print("\nUnivariate SVR analysis (Singular Tune - RandomizedSearch) complete.") # Updated print

# Visualization
print("\nGenerating SVR forecast plot...")
plt.figure(figsize=(15, 7))
plt.plot(y_train_level_initial.index, y_train_level_initial.values, 'k-', label='Training Data', alpha=0.7)
plt.plot(y_test_level.index, y_test_level.values, color='gray', linestyle='--', label='Actual Test Data', alpha=0.7)
colors = plt.cm.viridis(np.linspace(0, 0.8, len(results_summary_svr)))
for i, h in enumerate(svr_results_df.index):
    horizon_data = all_predictions_svr[h]
    num_preds = len(horizon_data['preds'])
    start_idx = h - 1
    end_idx = start_idx + num_preds
    forecast_dates = y_test_level.index[start_idx:end_idx]
    if len(forecast_dates) == num_preds: # Basic check
        plt.plot(forecast_dates, horizon_data['preds'], color=colors[i], linestyle='-', label=f'{h}-Step Forecast')

plt.title('SVR Model: Actual vs Forecasted House Index')
plt.ylabel('House Index')
plt.xlabel('Date')
plt.legend()
plt.grid(True, alpha=0.3)
display_start_date = pd.to_datetime('2015-01-01') # Or adjust as needed
plot_end_date = y_test_level.index.max()
plt.xlim(display_start_date, plot_end_date)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('svr_forecast_comparison_singletune.png', dpi=300)
plt.show()

### 4.2 SVR Forecast

In [ ]:
print("\nGenerating SVR forecast plot for selected horizons (1, 4, 8)...")

# --- Check if variables exist (optional but good practice) ---
required_vars = ['y_train_level_initial', 'y_test_level', 'all_predictions_svr', 'max_horizon']
if not all(var in globals() for var in required_vars):
    raise NameError("One or more required variables (y_train_level_initial, y_test_level, all_predictions_svr, max_horizon) not found. Please run the SVR forecasting script first.")

# --- Configuration ---
# *** EDITED: Define specific horizons to plot ***
horizons_to_plot = [1, 4, 8]
# Filter to ensure requested horizons exist in the results and are <= max_horizon
horizons_to_plot = [h for h in horizons_to_plot if h <= max_horizon and h in all_predictions_svr]

if not horizons_to_plot:
     raise ValueError(f"None of the specified horizons {horizons_to_plot} are available in the results.")

print(f"Plotting horizons: {horizons_to_plot}")

# --- Prepare Data & Plot ---
plt.figure(figsize=(15, 7))

# Plot historical data
plt.plot(y_train_level_initial.index, y_train_level_initial.values, 'k-', label='Training Data', alpha=0.7, linewidth=1.5)
plt.plot(y_test_level.index, y_test_level.values, color='gray', linestyle='--', label='Actual Test Data', alpha=0.7, linewidth=1.5)

# Define colors and markers for the selected horizons
colors = plt.cm.viridis(np.linspace(0, 0.8, len(horizons_to_plot)))
markers = ['o', 's', '^'] # Example markers

# *** EDITED: Loop through specified horizons_to_plot ***
for i, h in enumerate(horizons_to_plot):
    # Check if data exists for this specific horizon
    if h in all_predictions_svr and 'preds' in all_predictions_svr[h] and len(all_predictions_svr[h]['preds']) > 0:
        horizon_data = all_predictions_svr[h]
        preds_h = horizon_data['preds']
        num_preds = len(preds_h)

        # Calculate corresponding dates
        start_idx = h - 1
        end_idx = start_idx + num_preds
        # Ensure indices are within bounds of y_test_level.index
        if start_idx < 0 or end_idx > len(y_test_level.index):
             print(f"Warning: Index out of bounds for horizon {h}. Skipping plot for this horizon.")
             continue

        forecast_dates = y_test_level.index[start_idx:end_idx]

        # Double check length consistency after slicing
        if len(forecast_dates) == num_preds:
            plt.plot(forecast_dates, preds_h,
                     color=colors[i % len(colors)], # Use modulo for safety
                     marker=markers[i % len(markers)],
                     linestyle='-',
                     linewidth=1.2,
                     markersize=4,
                     label=f'{h}-Step Forecast')
        else:
             print(f"Warning: Length mismatch between predictions ({num_preds}) and dates ({len(forecast_dates)}) for horizon {h}. Skipping plot.")
    else:
        print(f"Warning: No prediction data found for horizon {h}.")


# --- Formatting ---
# *** EDITED: Updated title ***
plt.title('SVR Model: Actual vs Forecasted House Index (Horizons 1, 4, 8)')
plt.ylabel('House Index', fontsize=12)
plt.xlabel('Date', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

# Set x-axis limits (adjust start date if needed)
display_start_date = pd.to_datetime('2015-01-01')
# Find the latest date among the plotted horizons or test data
max_plot_dates = [forecast_dates.max() for h in horizons_to_plot if h in all_predictions_svr and 'preds' in all_predictions_svr[h] and len(all_predictions_svr[h]['preds']) > 0 and len(y_test_level.index[h-1:h-1+len(all_predictions_svr[h]['preds'])]) == len(all_predictions_svr[h]['preds'])]
plot_end_date = max(max_plot_dates) if max_plot_dates else y_test_level.index.max()

plt.xlim(display_start_date, plot_end_date + pd.DateOffset(months=1)) # Add padding

# Format date axis
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.gca().xaxis.set_minor_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45, ha='right')

plt.tight_layout()

# Save and Show Plot
try:
    # *** EDITED: Updated filename ***
    plt.savefig('svr_forecast_comparison_1_4_8.png', dpi=300, bbox_inches='tight')
    print("Plot saved as 'svr_forecast_comparison_1_4_8.png'")
except Exception as e:
    print(f"Error saving plot: {e}")

plt.show()

print("\n--- SVR Visualization Complete ---")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
# *** EDITED: Import RandomizedSearchCV and distributions ***
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from scipy.stats import uniform, loguniform # For parameter distributions
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
import time

# Ignore common warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

print("--- Starting Final SVR Model Training and Forecasting (RandomizedSearch Tuning - User Specific) ---") # Title updated

# --- Configuration ---
# Use the corrected input file
file_path = 'Preselect.csv'
target_pct_change_col = 'House_Index_pct_change'
target_level_col = 'House_Index'
date_col = 'Date'

# Final training end date and forecast steps
final_train_end_date_str = '2024-10-01'
num_forecast_steps = 8

# SVR Specific Configurations
max_lag = 3       # Use lags 1, 2, 3 as features
n_cv_splits = 9   # Number of CV splits for final tuning

# --- *** USER SPECIFIED SVR Parameter Space for RandomizedSearch *** ---
# This is the setup that yielded the best results (MSE 384) for the user previously
n_tuning_iterations = 10000 # Number of parameter settings to sample

svr_param_dist = {
    # Use the specific np.arange list for C as requested by user's successful run
    'C': np.arange(1, 101, 0.1),
    # Use the uniform distribution for epsilon as requested by user's successful run
    'epsilon': uniform(loc=0.0001, scale=0.1) # Samples uniformly between 0.0001 and 0.1001
}
print(f"Parameter space defined for RandomizedSearchCV:")
print(f"  C: np.arange(1, 101, 0.1) ({len(svr_param_dist['C'])} discrete values)")
print(f"  Epsilon: uniform(0.0001, 0.1001)")
print(f"RandomizedSearchCV will sample {n_tuning_iterations:,} combinations using {n_cv_splits} folds.")
print(f"Total SVR fits = {n_tuning_iterations * n_cv_splits:,}")
print("*** WARNING: This tuning process with 10k iterations may still take a very long time! ***")
# --- *** End Parameter Definition *** ---


# --- 1. Load and Prepare Data ---
print(f"Loading data from '{file_path}'...")
try:
    # Load data, ensuring Date column is parsed and set as index
    df = pd.read_csv(file_path, index_col=date_col, parse_dates=True)
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')] # Remove unnamed index columns
    df.sort_index(inplace=True)
    print("Data loaded and indexed.")
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
    raise
except Exception as e:
    print(f"Error loading {file_path}: {e}")
    raise

# Select only necessary columns: lagged target features (up to max_lag), target pct change, target level
lagged_target_features = []
for lag in range(1, max_lag + 1): # Use configured max_lag
    col_name = f"{target_pct_change_col}_lag{lag}"
    if col_name in df.columns:
        lagged_target_features.append(col_name)
    else:
        # Raise error if specifically required lags are missing
        raise ValueError(f"Required lag feature '{col_name}' not found in {file_path}.")

required_cols = lagged_target_features + [target_pct_change_col, target_level_col]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df_selected = df[required_cols].copy()

# Handle Missing Values (Should be minimal if using corrected file)
initial_rows = len(df_selected)
df_selected.dropna(inplace=True)
rows_after_na_drop = len(df_selected)
if initial_rows > rows_after_na_drop:
    print(f"Dropped {initial_rows - rows_after_na_drop} rows containing NaNs.")
print(f"Shape after dropping NaNs: {df_selected.shape}")
if df_selected.empty:
    raise ValueError("DataFrame empty after selecting columns and dropping NaNs.")

# Define final X (features), y (targets)
X = df_selected[lagged_target_features]
y_pct_change = df_selected[target_pct_change_col]
y_level = df_selected[target_level_col] # This is the full historical y_level after NaN drop
feature_names = X.columns.tolist() # Store feature names

# --- 2. Define Final Training Set ---
# Use all data up to the specified end date for final training/tuning
try:
    final_train_end_date = pd.to_datetime(final_train_end_date_str)
    if final_train_end_date not in X.index:
         print(f"Warning: Exact date {final_train_end_date_str} not found.")
         final_train_end_date = X[X.index <= final_train_end_date].index.max()
         print(f"Using last available date: {final_train_end_date.strftime('%Y-%m-%d')}")
except Exception as e:
    print(f"Error processing final_train_end_date: {e}")
    raise

X_train_final = X[X.index <= final_train_end_date]
y_train_pct_change_final = y_pct_change[y_pct_change.index <= final_train_end_date]
# y_train_level_final is defined here but not strictly needed later if we use y_level
y_train_level_final = y_level[y_level.index <= final_train_end_date]

if X_train_final.empty:
    raise ValueError("Final training set is empty.")
print(f"Final training data shape: {X_train_final.shape}")
print(f"Final training period ends: {final_train_end_date.strftime('%Y-%m-%d')}")

# --- 3. Final Scaling ---
# Scale features based on the final training set
final_scaler = StandardScaler()
X_train_final_scaled = final_scaler.fit_transform(X_train_final)
print("Final scaler fitted on all training data.")
# Store means/stds for iterative forecast scaling
final_feature_means = final_scaler.mean_
final_feature_stds = final_scaler.scale_
# Handle potential zero std dev
final_feature_stds = np.where(final_feature_stds == 0, 1.0, final_feature_stds)

# --- 4. Final SVR Hyperparameter Tuning (using RandomizedSearchCV) ---
print(f"\nTuning SVR hyperparameters on final training data using RandomizedSearchCV (n_iter={n_tuning_iterations}, cv={n_cv_splits})...") # Updated print
tuning_start_time = time.time()
tscv = TimeSeriesSplit(n_splits=n_cv_splits)
# Use a large max_iter as before
svr_linear = SVR(kernel='linear', max_iter=200000)

# --- Instantiate RandomizedSearchCV ---
random_search = RandomizedSearchCV(
    estimator=svr_linear,
    param_distributions=svr_param_dist, # Use the specified dist/list combo
    n_iter=n_tuning_iterations, # Set number of iterations
    cv=tscv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1, # Set verbose=1 to see progress
    random_state=42 # For reproducibility
)
# Fit RandomizedSearchCV
random_search.fit(X_train_final_scaled, y_train_pct_change_final)
# --- End Edit ---

# --- Get best params from random_search object ---
best_params = random_search.best_params_
final_best_svr_c = best_params['C']
final_best_svr_epsilon = best_params['epsilon']
# --- End Edit ---

print(f"\nFinal SVR tuning complete ({time.time() - tuning_start_time:.2f}s).")
print(f"  Best Parameters Found:")
# Use appropriate formatting for potentially float C and epsilon
print(f"  Best C = {final_best_svr_c:.5f}")
print(f"  Best Epsilon = {final_best_svr_epsilon:.5f}")

# --- 5. Train Final SVR Model ---
print("\nTraining final SVR model with best parameters found...")
final_svr_model = SVR(kernel='linear', C=final_best_svr_c, epsilon=final_best_svr_epsilon, max_iter=200000)
final_svr_model.fit(X_train_final_scaled, y_train_pct_change_final)
print("Final SVR model trained.")

# --- 6. Prepare for Forecasting ---
# Use y_train_level_final here as it's guaranteed to be defined if section 2 ran
last_known_level = y_train_level_final.iloc[-1]
last_date = y_train_level_final.index[-1]

# Get the features corresponding to the last date
last_features_row = X.loc[[last_date]]
# Scale these features using the FINAL scaler
current_features_scaled_np = final_scaler.transform(last_features_row)

print(f"\nForecasting starting after {last_date.strftime('%Y-%m-%d')} using level {last_known_level:.4f}")

# --- 7. Iterative Forecasting Loop ---
forecast_results = []
current_pred_level = last_known_level

# Determine date frequency for generating future dates
inferred_freq = pd.infer_freq(X_train_final.index)
print(f"Inferred data frequency: {inferred_freq}")
if inferred_freq is None:
    print("Warning: Could not infer frequency. Assuming Monthly Start ('MS'). Adjust offset if needed.")
    # Assuming monthly if quarterly inference fails - adjust if needed
    date_offset = pd.tseries.offsets.DateOffset(months=1)
else:
    # Use inferred frequency (likely 'QS-OCT' or similar for quarterly)
    date_offset = pd.tseries.frequencies.to_offset(inferred_freq)
    print(f"Using offset: {date_offset}")


current_pred_date = last_date

for step in range(1, num_forecast_steps + 1):
    # Predict 1 step ahead based on *current* features
    svr_pred_pct = final_svr_model.predict(current_features_scaled_np)[0]

    # Calculate predicted level for this step
    current_pred_level = current_pred_level * (1 + svr_pred_pct)

    # Calculate the date for this forecast step
    current_pred_date = current_pred_date + date_offset
    forecast_date_str = current_pred_date.strftime('%Y-%m-%d')

    # Store results
    forecast_results.append({
        'Date': forecast_date_str,
        'SVR_Forecast': current_pred_level
    })

    # --- Feature Update for the NEXT step's prediction ---
    if step < num_forecast_steps:
        # Unscale current features
        current_features_unscaled = (current_features_scaled_np * final_feature_stds) + final_feature_means
        current_features_unscaled_dict = dict(zip(feature_names, current_features_unscaled[0]))

        # Create dict for next step's unscaled features
        next_features_unscaled_dict = current_features_unscaled_dict.copy()

        # Update lagged features based on the prediction just made
        for lag in range(max_lag, 0, -1): # Using max_lag = 3 here
            current_lag_col = f"{target_pct_change_col}_lag{lag}"
            if lag == 1:
                next_features_unscaled_dict[current_lag_col] = svr_pred_pct
            else:
                prev_lag_col = f"{target_pct_change_col}_lag{lag-1}"
                # Use the value *before* potential update in this inner loop
                next_features_unscaled_dict[current_lag_col] = current_features_unscaled_dict[prev_lag_col]

        # Convert updated features back to numpy array in correct order
        next_features_unscaled_np = np.array([next_features_unscaled_dict[name] for name in feature_names]).reshape(1, -1)

        # Scale the updated features using the FINAL scaler
        current_features_scaled_np = (next_features_unscaled_np - final_feature_means) / final_feature_stds
    # End feature update block
# End forecast loop

# --- 8. Print Forecasts ---
forecast_df = pd.DataFrame(forecast_results)
forecast_df['Date'] = pd.to_datetime(forecast_df['Date'])
forecast_df = forecast_df.set_index('Date')

print(f"\n--- SVR Forecasts for the next {num_forecast_steps} steps after {last_date.strftime('%Y-%m-%d')} ---")
print(forecast_df.round(4))

# --- 9. Visualization ---
print("\nGenerating final SVR forecast plot...")
plt.figure(figsize=(15, 7))

# Plot historical data (all historical data used for training)
# *** EDITED LINE BELOW: Use y_level directly and filter ***
plt.plot(y_level.index[y_level.index <= final_train_end_date], y_level[y_level.index <= final_train_end_date].values, 'k-', label='Historical Actual Data', linewidth=1.5)

# Combine last historical point with forecasts for smooth line
svr_plot_index = pd.Index([last_date]).union(forecast_df.index)
# Ensure concatenation works with potentially empty forecast_df if something went wrong
svr_forecast_series = forecast_df['SVR_Forecast'] if not forecast_df.empty else pd.Series(dtype=float)
svr_plot_values = pd.concat([pd.Series([last_known_level], index=[last_date]), svr_forecast_series])


# Plot forecast
plt.plot(svr_plot_index, svr_plot_values, 'm-s', label=f'SVR Forecast (C={final_best_svr_c:.2f}, eps={final_best_svr_epsilon:.4f})', linewidth=1.5, markersize=4) # Magenta squares

# Formatting
plt.title(f'House Index: Historical Data and {num_forecast_steps}-Step SVR Forecast after {last_date.strftime("%Y-%m-%d")}', fontsize=14)
plt.ylabel('House Index', fontsize=12)
plt.xlabel('Date', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

# Set x-axis limits
display_history_years = 5
viz_start_date = last_date - pd.DateOffset(years=display_history_years)
viz_end_date = forecast_df.index.max() + pd.DateOffset(months=3) if not forecast_df.empty else last_date + pd.DateOffset(months=3) # Add padding
plt.xlim(viz_start_date, viz_end_date)

# Format date axis
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.gca().xaxis.set_minor_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45, ha='right')

plt.tight_layout()

# Save and Show Plot
try:
    # Include best params in filename for easy reference
    plt.savefig(f'final_svr_forecast_randomsearch10k_C{final_best_svr_c:.1f}_eps{final_best_svr_epsilon:.4f}.png', dpi=300, bbox_inches='tight') # Updated filename
    print(f"Plot saved.")
except Exception as e:
    print(f"Error saving plot: {e}")

plt.show()

print("\n--- Final SVR forecasting process complete. ---")


# 5. LSTM

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
# Deep Learning imports
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input, Dropout # Added Dropout
from tensorflow.keras.callbacks import EarlyStopping
import keras_tuner as kt # Import Keras Tuner
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
import time
import os # For managing tuner directories
import shutil # For potentially clearing tuner directory

# --- Configuration ---
# Set random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

warnings.filterwarnings('ignore')

# Input file containing features and original index level
file_path = 'Preselect.csv' # Use the corrected file
# Target variable (percentage change - for prediction)
target_pct_change_col = 'House_Index_pct_change'
# Original level column (for evaluation and iterative prediction)
target_level_col = 'House_Index'
# Date column name
date_col = 'Date'

# Define the training and testing periods
train_end_date_str = '2015-12-31' # Adjust as needed
test_start_date_str = '2016-01-01' # Adjust as needed
final_historical_date_str = '2024-10-01' # Last date of real historical data to use

# LSTM Parameters
n_timesteps = 4   # Lookback window (how many past steps LSTM sees)
n_outputs = 8     # Number of steps to predict ahead (h=1 to 8)

# --- Hyperparameter Tuning Configuration ---
# NOTE: Tuning inside the loop is VERY time-consuming!
tune_dropout = True        # Option to tune dropout rate
tuner_max_trials = 8      # Number of hyperparameter combinations to try per step (User updated)
tuner_epochs = 15          # Max epochs *during* the tuning search for each trial
final_model_epochs = 50    # Max epochs for training the *best* model after tuning
lstm_batch_size = 16       # Batch size for LSTM training
lstm_patience = 10         # Patience for EarlyStopping in final model training

print(f"--- Running Multi-Output LSTM Evaluation (Predicting Pct Change Vector) ---")
print(f"Using Bayesian Optimization Tuner: Max Trials={tuner_max_trials}, Tuner Epochs={tuner_epochs}")
print(f"Tuning Dropout: {tune_dropout}")
print(f"LSTM Settings: Timesteps={n_timesteps}, Outputs={n_outputs}, Final Epochs={final_model_epochs}")

# --- Helper Functions ---
def mean_absolute_percentage_error_np(y_true, y_pred):
    """Numpy implementation of MAPE"""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    if np.sum(mask) == 0: return np.nan
    # Ensure y_true[mask] is not empty before division
    if len(y_true[mask]) == 0: return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def directional_accuracy_np(y_true, y_pred, y_true_prev):
    """Numpy implementation of Directional Accuracy"""
    y_true, y_pred, y_true_prev = np.array(y_true), np.array(y_pred), np.array(y_true_prev)
    min_len = min(len(y_true), len(y_pred), len(y_true_prev))
    if min_len == 0: return np.nan
    y_true, y_pred, y_true_prev = y_true[:min_len], y_pred[:min_len], y_true_prev[:min_len]

    actual_diff = np.sign(y_true - y_true_prev)
    pred_diff = np.sign(y_pred - y_true_prev)
    correct_direction = (actual_diff == pred_diff).astype(int)
    return np.mean(correct_direction) * 100

def create_multi_output_sequences(X_data, y_data, n_timesteps, n_outputs):
    """Creates sequences for multi-output LSTM"""
    X_seq, y_seq, seq_indices = [], [], []
    for i in range(len(X_data) - n_timesteps - n_outputs + 1):
        X_seq.append(X_data[i:(i + n_timesteps)])
        y_seq.append(y_data[(i + n_timesteps):(i + n_timesteps + n_outputs)])
        seq_indices.append(i + n_timesteps - 1)
    return np.array(X_seq), np.array(y_seq), np.array(seq_indices)

# --- Modified build_multi_lstm_model function for Keras Tuner ---
def build_multi_lstm_model(hp, n_timesteps, n_features, n_outputs, tune_dropout_flag):
    """Builds the multi-output LSTM model with hyperparameters."""
    model = Sequential()
    model.add(Input(shape=(n_timesteps, n_features)))
    # Tune LSTM units (User updated range)
    hp_units = hp.Int('units', min_value=8, max_value=128, step=4) # User updated range
    model.add(LSTM(hp_units, activation='relu', return_sequences=False))

    # Optional Dropout Tuning
    if tune_dropout_flag:
        hp_dropout = hp.Float('dropout', min_value=0.0, max_value=0.5, step=0.1)
        model.add(Dropout(hp_dropout))

    model.add(Dense(n_outputs))

    # Tune learning rate (User updated choices)
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-3, 5e-3, 5e-4, 5e-5, 1e-4, 1e-5])

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
                  loss='mse')
    return model

# --- 1. Load Data ---
print(f"Loading data from '{file_path}'...")
try:
    df = pd.read_csv(file_path, index_col=date_col, parse_dates=True)
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    df.sort_index(inplace=True)
    print("Data loaded and indexed.")
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
    raise

# --- 2. Prepare Data ---
# Filter data to include only up to final historical date
final_historical_date = pd.to_datetime(final_historical_date_str)
df_filtered = df[df.index <= final_historical_date].copy()
print(f"Data filtered to include dates up to {final_historical_date_str}. Shape: {df_filtered.shape}")
if df_filtered.empty:
    raise ValueError("DataFrame is empty after filtering by final historical date.")

feature_cols = [col for col in df_filtered.columns if col not in [target_level_col, target_pct_change_col]]
if not feature_cols: raise ValueError("No feature columns identified.")
required_targets = [target_pct_change_col, target_level_col]
missing_targets = [col for col in required_targets if col not in df_filtered.columns]
if missing_targets: raise ValueError(f"Missing target columns: {missing_targets}")

df_processed = df_filtered[feature_cols + required_targets].copy()
initial_rows = len(df_processed)
df_processed.dropna(inplace=True)
rows_after_na_drop = len(df_processed)
print(f"Dropped {initial_rows - rows_after_na_drop} rows containing NaNs.")
if df_processed.empty: raise ValueError("DataFrame empty after dropping NaNs.")

X_all = df_processed[feature_cols]
y_pct_change_all = df_processed[[target_pct_change_col]]
y_level_all = df_processed[[target_level_col]]
n_features = X_all.shape[1]
print(f"Using {n_features} features.")

# --- 3. Split Data (Temporal) ---
train_end_date = pd.to_datetime(train_end_date_str)
test_start_date = pd.to_datetime(test_start_date_str)

X_train = X_all[X_all.index <= train_end_date]
y_train_pct_change = y_pct_change_all[y_pct_change_all.index <= train_end_date]
y_train_level = y_level_all[y_level_all.index <= train_end_date]

X_test = X_all[X_all.index >= test_start_date]
y_test_pct_change = y_pct_change_all[y_pct_change_all.index >= test_start_date]
y_test_level = y_level_all[y_level_all.index >= test_start_date]

if X_test.empty or X_train.empty: raise ValueError("Training or Test set is empty after split.")
print(f"\nInitial training data shape: X={X_train.shape}, y={y_train_pct_change.shape}")
print(f"Test data shape: X={X_test.shape}, y={y_test_pct_change.shape}")

# --- 4. Scaling ---
print("Scaling features and target (fitting only on training data)...")
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train_pct_change)
y_test_scaled = scaler_y.transform(y_test_pct_change)
print("Scaling complete.")

# --- 5. Create Sequences ---
print(f"Creating sequences for the entire dataset (n_timesteps={n_timesteps}, n_outputs={n_outputs})...")
X_scaled_all = np.concatenate((X_train_scaled, X_test_scaled), axis=0)
y_scaled_all = np.concatenate((y_train_scaled, y_test_scaled), axis=0)
X_seq_all, y_seq_all, seq_indices_all = create_multi_output_sequences(
    X_scaled_all, y_scaled_all.flatten(), n_timesteps, n_outputs
)
seq_end_dates_all = X_all.index[seq_indices_all]
print(f"Total sequences created: {len(X_seq_all)}")

# --- 6. Split Sequences for Walk-Forward ---
first_pred_dates = X_all.index[seq_indices_all + 1]
test_seq_start_index = np.argmax(first_pred_dates >= test_start_date)
if test_seq_start_index == 0 and first_pred_dates[0] < test_start_date:
     raise ValueError("Could not find start of test sequences.")

X_seq_train = X_seq_all[:test_seq_start_index]
y_seq_train = y_seq_all[:test_seq_start_index]
X_seq_test = X_seq_all[test_seq_start_index:]
y_seq_test_actual_scaled = y_seq_all[test_seq_start_index:]
test_seq_end_input_dates = seq_end_dates_all[test_seq_start_index:]

print(f"Sequence Train shapes: X={X_seq_train.shape}, y={y_seq_train.shape}")
print(f"Sequence Test shapes: X={X_seq_test.shape}, y={y_seq_test_actual_scaled.shape}")
print(f"Number of test forecast origins: {len(X_seq_test)}")

# --- 7. Walk-Forward Validation ---
history_X = [x for x in X_seq_train]
history_y = [y for y in y_seq_train]
predictions_level = {h: [] for h in range(1, n_outputs + 1)}
actuals_level = {h: [] for h in range(1, n_outputs + 1)}
prev_actuals_level = {h: [] for h in range(1, n_outputs + 1)}
target_dates_level = {h: [] for h in range(1, n_outputs + 1)}

print("\nStarting walk-forward validation (LSTM with Bayesian Optimization Tuning)...")
start_walk_time = time.time()

# Define tuner directory outside the loop
tuner_dir = 'lstm_bo_tuner_dir'

for t in range(len(X_seq_test)):
    end_input_date = test_seq_end_input_dates[t]
    print(f"\nProcessing Origin {t + 1}/{len(X_seq_test)} (Input Ends: {end_input_date.date()})...")
    start_step_time = time.time()

    current_X_train = np.array(history_X)
    current_y_train = np.array(history_y)

    # --- 7.1 Tune Model ---
    print(f"   Tuning hyperparameters (max_trials={tuner_max_trials})...")
    # Pass the tune_dropout flag to the build function
    build_fn = lambda hp: build_multi_lstm_model(hp, n_timesteps, n_features, n_outputs, tune_dropout)
    tuner = kt.BayesianOptimization(
        build_fn,
        objective='val_loss',
        max_trials=tuner_max_trials,
        executions_per_trial=1,
        directory=tuner_dir,
        project_name='lstm_bo_tuning', # Reuse the same project name
        overwrite=True # Overwrite previous trial results for this step
    )

    tuner_early_stopping = EarlyStopping(monitor='val_loss', patience=5)

    try:
        tuner.search(current_X_train, current_y_train,
                     epochs=tuner_epochs,
                     batch_size=lstm_batch_size,
                     validation_split=0.2,
                     callbacks=[tuner_early_stopping],
                     verbose=0)

        best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
        print(f"   Best HPs found: Units={best_hps.get('units')}, LR={best_hps.get('learning_rate')}" +
              (f", Dropout={best_hps.get('dropout'):.1f}" if tune_dropout else "")) # Print dropout if tuned
        # Build the best model found by the tuner
        model = tuner.hypermodel.build(best_hps)

    except Exception as e:
        print(f"   ERROR during Keras Tuner search for step {t+1}: {e}")
        print(f"   Skipping prediction for this origin.")
        history_X.append(X_seq_test[t])
        history_y.append(y_seq_test_actual_scaled[t])
        tf.keras.backend.clear_session()
        continue

    # --- 7.2 Train Final Model for this step ---
    print(f"   Training final model for step {t+1} (max_epochs={final_model_epochs})...")
    final_early_stopping = EarlyStopping(monitor='loss', patience=lstm_patience, restore_best_weights=True)
    try:
        model.fit(current_X_train, current_y_train,
                  epochs=final_model_epochs, verbose=0, batch_size=lstm_batch_size,
                  callbacks=[final_early_stopping])
    except Exception as e:
        print(f"   ERROR during final model training for step {t+1}: {e}")
        print(f"   Skipping prediction for this origin.")
        history_X.append(X_seq_test[t])
        history_y.append(y_seq_test_actual_scaled[t])
        tf.keras.backend.clear_session()
        continue

    # --- 7.3 Predict ---
    try:
        current_X_test_seq = X_seq_test[t].reshape((1, n_timesteps, n_features))
        y_pred_scaled_vector = model.predict(current_X_test_seq, verbose=0)[0]
    except Exception as e:
        print(f"   ERROR during prediction for step {t+1}: {e}")
        print(f"   Skipping prediction storage for this origin.")
        history_X.append(X_seq_test[t])
        history_y.append(y_seq_test_actual_scaled[t])
        tf.keras.backend.clear_session()
        continue

    # --- 7.4 Inverse Transform and Store ---
    try:
        y_pred_pct_change_vector = scaler_y.inverse_transform(y_pred_scaled_vector.reshape(1, -1)).flatten()
        last_actual_level = y_level_all.loc[end_input_date].iloc[0]
    except KeyError:
        print(f"Warning: Cannot find level at {end_input_date.date()}. Skipping origin.")
        history_X.append(X_seq_test[t])
        history_y.append(y_seq_test_actual_scaled[t])
        tf.keras.backend.clear_session()
        continue
    except Exception as e:
         print(f"   ERROR during inverse transform setup for step {t+1}: {e}")
         print(f"   Skipping prediction storage for this origin.")
         history_X.append(X_seq_test[t])
         history_y.append(y_seq_test_actual_scaled[t])
         tf.keras.backend.clear_session()
         continue

    current_level = last_actual_level
    current_date = end_input_date
    freq = pd.infer_freq(X_all.index)

    for h in range(1, n_outputs + 1):
        pred_pct_change = y_pred_pct_change_vector[h-1]
        current_level = current_level * (1 + pred_pct_change)
        try:
             if freq is None: raise ValueError("Cannot infer frequency")
             target_date = end_input_date + pd.tseries.frequencies.to_offset(freq) * h
        except Exception:
             target_date = end_input_date + pd.DateOffset(months=h) # Fallback

        if target_date <= final_historical_date:
            predictions_level[h].append(current_level)
            target_dates_level[h].append(target_date)
            if target_date in y_level_all.index:
                actuals_level[h].append(y_level_all.loc[target_date].iloc[0])
                prev_actual_date = target_date - (pd.tseries.frequencies.to_offset(freq) if freq else pd.DateOffset(months=1))
                if prev_actual_date in y_level_all.index:
                     prev_actuals_level[h].append(y_level_all.loc[prev_actual_date].iloc[0])
                else:
                     actuals_level[h].pop()
                     predictions_level[h].pop()
                     target_dates_level[h].pop()
            else:
                 predictions_level[h].pop()
                 target_dates_level[h].pop()

    # --- 7.5 Update History ---
    history_X.append(X_seq_test[t])
    history_y.append(y_seq_test_actual_scaled[t])

    step_time = time.time() - start_step_time
    print(f"   Step {t + 1} finished in {step_time:.2f} seconds.")
    # Clear TensorFlow session memory
    tf.keras.backend.clear_session()

walk_time = time.time() - start_walk_time
print(f"\nWalk-forward validation complete in {walk_time:.2f} seconds ({walk_time/60:.2f} minutes).")

# --- 8. Calculate Metrics for Each Horizon ---
print("\nCalculating performance metrics...")
final_metrics = []
plot_dfs = {}

for h in range(1, n_outputs + 1):
    preds = np.array(predictions_level[h])
    actuals = np.array(actuals_level[h])
    prev_actuals = np.array(prev_actuals_level[h])
    indices = pd.to_datetime(target_dates_level[h])

    min_len = min(len(preds), len(actuals), len(prev_actuals))
    if len(preds) != min_len or len(actuals) != min_len or len(prev_actuals) != min_len:
         print(f"Warning: Length mismatch for horizon {h}. Trimming arrays.")
         preds = preds[:min_len]
         actuals = actuals[:min_len]
         prev_actuals = prev_actuals[:min_len]
         indices = indices[:min_len]

    if min_len > 0:
        # *** Calculate MSE ***
        mse = mean_squared_error(actuals, preds)
        mae = mean_absolute_error(actuals, preds)
        rmse = np.sqrt(mse) # Calculate RMSE from MSE
        mape = mean_absolute_percentage_error_np(actuals, preds)
        da = directional_accuracy_np(actuals, preds, prev_actuals)
        # *** Store MSE ***
        final_metrics.append({'Forecast Horizon (Steps)': h, 'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'DA': da})
        plot_dfs[h] = pd.DataFrame({'Actual_Level': actuals, 'Predicted_Level': preds}, index=indices)
    else:
        print(f"No valid predictions/actuals found for horizon h={h}. Skipping metrics.")
        # *** Add MSE placeholder ***
        final_metrics.append({'Forecast Horizon (Steps)': h, 'MSE': np.nan, 'RMSE': np.nan, 'MAE': np.nan, 'MAPE': np.nan, 'DA': np.nan})

metrics_df = pd.DataFrame(final_metrics)
# *** Reorder columns to put MSE first ***
metric_cols_order = ['Forecast Horizon (Steps)', 'MSE', 'RMSE', 'MAE', 'MAPE', 'DA']
existing_cols = [col for col in metric_cols_order if col in metrics_df.columns]
metrics_df = metrics_df[existing_cols]


print("\n--- Multi-Output LSTM Performance Metrics (Bayesian Tuned) ---")
print(metrics_df.to_string(index=False, float_format="%.3f"))

valid_metrics = metrics_df.dropna()
if not valid_metrics.empty:
     avg_metrics_lstm = valid_metrics.drop(columns='Forecast Horizon (Steps)').mean()
     print(f"\n--- Average LSTM Performance Across Valid Horizons (Bayesian Tuned) ---")
     # *** Add Avg MSE printout ***
     print(f"  Avg MSE: {avg_metrics_lstm['MSE']:.3f}")
     print(f"  Avg RMSE: {avg_metrics_lstm['RMSE']:.3f}")
     print(f"  Avg MAE: {avg_metrics_lstm['MAE']:.3f}")
     print(f"  Avg MAPE (%): {avg_metrics_lstm['MAPE']:.3f}")
     print(f"  Avg DA (%): {avg_metrics_lstm['DA']:.3f}")
else:
     print("\nNo valid horizons found to calculate average performance.")


# --- 9. Plotting Selected Horizons ---
# (Plotting code remains the same)
print("\n--- Plotting Results for Selected Horizons (h=1, 4, 8) ---")
plt.figure(figsize=(15, 7))
horizons_to_plot = [1, 4, 8]
min_plot_date = min((df.index.min() for h, df in plot_dfs.items() if h in horizons_to_plot and not df.empty), default=train_end_date)
max_plot_date = max((df.index.max() for h, df in plot_dfs.items() if h in horizons_to_plot and not df.empty), default=final_historical_date)
plot_start_hist_date = min_plot_date - pd.DateOffset(years=2)
actual_data_subset = y_level_all[(y_level_all.index >= plot_start_hist_date) & (y_level_all.index <= max_plot_date)]
plt.plot(actual_data_subset.index, actual_data_subset[target_level_col], label='Actual House Index', color='black', linewidth=1.5, alpha=0.7)
colors = plt.cm.viridis(np.linspace(0, 0.8, len(horizons_to_plot)))
markers = ['o', 's', '^']
for i, h in enumerate(horizons_to_plot):
    if h in plot_dfs and not plot_dfs[h].empty:
        results_df = plot_dfs[h]
        plt.plot(results_df.index, results_df['Predicted_Level'],
                 label=f'LSTM Predicted Level (h={h})', color=colors[i],
                 marker=markers[i % len(markers)], linestyle='--', markersize=4)
plt.title('Multi-Output LSTM Forecasts vs Actual Level (Iterative Bayesian Tuning)')
plt.xlabel('Date')
plt.ylabel('House Index')
plt.legend()
plt.grid(True, alpha=0.4)
plt.xlim(plot_start_hist_date, max_plot_date + pd.DateOffset(months=1))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
try:
    plt.savefig('lstm_multioutput_bayes_tuned_forecast_comparison.png', dpi=300, bbox_inches='tight')
    print("Plot saved as 'lstm_multioutput_bayes_tuned_forecast_comparison.png'")
except Exception as e:
    print(f"Error saving plot: {e}")
plt.show()

print(f"\n--- Completed Tuned Multi-Output LSTM Run ---")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
# Deep Learning imports
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import keras_tuner as kt
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
import time
import os
import shutil

# --- Configuration ---
# Set random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

warnings.filterwarnings('ignore')

# Input file containing features and original index level
file_path = 'Preselect.csv' # Use the corrected file
# Target variable (percentage change - for prediction)
target_pct_change_col = 'House_Index_pct_change'
# Original level column (for evaluation and iterative prediction)
target_level_col = 'House_Index'
# Date column name
date_col = 'Date'

# Final training end date and forecast steps
final_train_end_date_str = '2024-10-01' # Use all data UP TO this date for training/tuning
num_forecast_steps = 8 # Predict 8 steps ahead

# LSTM Parameters
n_timesteps = 4   # Lookback window (how many past steps LSTM sees)
n_outputs = num_forecast_steps # Predict the required number of steps

# --- Hyperparameter Tuning Configuration (Performed ONCE) ---
tune_dropout = True        # Option to tune dropout rate
tuner_max_trials = 15      # Number of hyperparameter combinations to try (User updated)
tuner_epochs = 15          # Max epochs *during* the tuning search for each trial
# --- Final Model Training Configuration ---
final_model_epochs = 50    # Max epochs for training the *best* model after tuning
lstm_batch_size = 16       # Batch size for LSTM training
lstm_patience = 10         # Patience for EarlyStopping in final model training

print(f"--- Running Final LSTM Forecast Generation ---")
print(f"--- Tuning ONCE on data up to {final_train_end_date_str} ---")
print(f"Using Bayesian Optimization Tuner: Max Trials={tuner_max_trials}, Tuner Epochs={tuner_epochs}")
print(f"Tuning Dropout: {tune_dropout}")
print(f"LSTM Settings: Timesteps={n_timesteps}, Outputs={n_outputs}, Final Epochs={final_model_epochs}")

# --- Helper Functions ---
def create_multi_output_sequences(X_data, y_data, n_timesteps, n_outputs):
    """Creates sequences for multi-output LSTM"""
    X_seq, y_seq, seq_indices = [], [], []
    # Ensure loop doesn't go out of bounds for features or targets
    # Need n_timesteps history for X, and n_outputs future for y
    for i in range(len(X_data) - n_timesteps - n_outputs + 1):
        X_seq.append(X_data[i:(i + n_timesteps)])
        y_seq.append(y_data[(i + n_timesteps):(i + n_timesteps + n_outputs)])
        # Store the index of the *end* of the input sequence (i + n_timesteps - 1)
        seq_indices.append(i + n_timesteps - 1)
    return np.array(X_seq), np.array(y_seq), np.array(seq_indices)

# --- Build Model Function for Keras Tuner ---
def build_multi_lstm_model(hp, n_timesteps, n_features, n_outputs, tune_dropout_flag):
    """Builds the multi-output LSTM model with hyperparameters."""
    model = Sequential()
    model.add(Input(shape=(n_timesteps, n_features)))
    # Tune LSTM units (User updated range)
    hp_units = hp.Int('units', min_value=8, max_value=128, step=4) # User updated range
    model.add(LSTM(hp_units, activation='relu', return_sequences=False))

    # Optional Dropout Tuning
    if tune_dropout_flag:
        hp_dropout = hp.Float('dropout', min_value=0.0, max_value=0.5, step=0.1)
        model.add(Dropout(hp_dropout))

    model.add(Dense(n_outputs))

    # Tune learning rate (User updated choices)
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-3, 5e-3, 5e-4, 5e-5, 1e-4, 1e-5])

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
                  loss='mse')
    return model

# --- 1. Load Data ---
print(f"\nLoading data from '{file_path}'...")
# Filter data after 10/01/2024
try:
    df = pd.read_csv(file_path, index_col=date_col, parse_dates=True)
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    df.sort_index(inplace=True)\
    # Filter data after 10/01/2024
    df = df[df.index <= pd.to_datetime(final_train_end_date_str)]
    print("Data loaded and indexed.")
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
    raise

# --- 2. Prepare Data ---
# Filter data to include only up to final historical date
final_historical_date = pd.to_datetime(final_train_end_date_str) # Use final_train_end_date_str
df_filtered = df[df.index <= final_historical_date].copy()
print(f"Data filtered to include dates up to {final_train_end_date_str}. Shape: {df_filtered.shape}")
if df_filtered.empty:
    raise ValueError("DataFrame is empty after filtering by final historical date.")

feature_cols = [col for col in df_filtered.columns if col not in [target_level_col, target_pct_change_col]]
if not feature_cols: raise ValueError("No feature columns identified.")
required_targets = [target_pct_change_col, target_level_col]
missing_targets = [col for col in required_targets if col not in df_filtered.columns]
if missing_targets: raise ValueError(f"Missing target columns: {missing_targets}")

df_processed = df_filtered[feature_cols + required_targets].copy()
initial_rows = len(df_processed)
df_processed.dropna(inplace=True)
rows_after_na_drop = len(df_processed)
print(f"Dropped {initial_rows - rows_after_na_drop} rows containing NaNs.")
if df_processed.empty: raise ValueError("DataFrame empty after dropping NaNs.")

# Define final training data (all available historical data after NaN drop)
X_train_final = df_processed[feature_cols]
y_train_pct_change_final = df_processed[[target_pct_change_col]]
y_train_level_final = df_processed[[target_level_col]]
n_features = X_train_final.shape[1]
print(f"Using {n_features} features for final training.")
print(f"Final training data shape: X={X_train_final.shape}, y={y_train_pct_change_final.shape}")

# --- 3. Final Scaling ---
print("\nScaling features and target (fitting on final training data)...")
final_scaler_X = StandardScaler()
X_train_final_scaled = final_scaler_X.fit_transform(X_train_final)

final_scaler_y = StandardScaler()
y_train_final_scaled = final_scaler_y.fit_transform(y_train_pct_change_final)
print("Scaling complete.")

# --- 4. Create Sequences for Tuning/Training ---
print(f"Creating sequences for the final training data (n_timesteps={n_timesteps}, n_outputs={n_outputs})...")
# Need enough data for at least one sequence
if len(X_train_final_scaled) < n_timesteps + n_outputs:
    raise ValueError(f"Not enough data ({len(X_train_final_scaled)} rows) to create sequences with n_timesteps={n_timesteps} and n_outputs={n_outputs}.")

X_seq_train_final, y_seq_train_final, _ = create_multi_output_sequences(
    X_train_final_scaled, y_train_final_scaled.flatten(), n_timesteps, n_outputs
)
print(f"Final Training sequences created: X={X_seq_train_final.shape}, y={y_seq_train_final.shape}")
if X_seq_train_final.shape[0] == 0:
    raise ValueError("No training sequences were generated. Check data length and sequence parameters.")


# --- 5. Final Hyperparameter Tuning (Performed ONCE) ---
print(f"\nTuning hyperparameters ONCE on final training sequences (max_trials={tuner_max_trials})...")
tuner_dir = 'lstm_bo_tuner_final' # New directory for final tuning results
# Optional: Clean previous results
if os.path.exists(tuner_dir):
    print(f"Cleaning previous tuner directory: {tuner_dir}")
    shutil.rmtree(tuner_dir)

build_fn = lambda hp: build_multi_lstm_model(hp, n_timesteps, n_features, n_outputs, tune_dropout)
tuner = kt.BayesianOptimization(
    build_fn,
    objective='val_loss',
    max_trials=tuner_max_trials,
    executions_per_trial=1,
    directory=tuner_dir,
    project_name='lstm_final_tuning', # Single project name
    overwrite=True
)

tuner_early_stopping = EarlyStopping(monitor='val_loss', patience=5)
tuning_start_time = time.time()

try:
    # Tune using the final training sequences
    tuner.search(X_seq_train_final, y_seq_train_final,
                 epochs=tuner_epochs,
                 batch_size=lstm_batch_size,
                 validation_split=0.2, # Use last 20% of final train sequences for validation
                 callbacks=[tuner_early_stopping],
                 verbose=1) # Show tuner progress

    # Get the single best set of hyperparameters
    best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
    print(f"\nFinal Tuning Complete ({time.time() - tuning_start_time:.2f}s).")
    print(f"   Best HPs found: Units={best_hps.get('units')}, LR={best_hps.get('learning_rate')}" +
          (f", Dropout={best_hps.get('dropout'):.1f}" if tune_dropout else ""))

    # Build the final model structure using the best HPs
    final_model = tuner.hypermodel.build(best_hps)

except Exception as e:
    print(f"   ERROR during final Keras Tuner search: {e}")
    raise # Stop execution if final tuning fails

# --- 6. Train Final Model ---
print(f"\nTraining final model with best HPs (max_epochs={final_model_epochs})...")
final_early_stopping = EarlyStopping(monitor='loss', patience=lstm_patience, restore_best_weights=True)
training_start_time = time.time()
try:
    # Train on the full final sequence set
    final_model.fit(X_seq_train_final, y_seq_train_final,
              epochs=final_model_epochs, verbose=1, batch_size=lstm_batch_size,
              callbacks=[final_early_stopping])
    print(f"Final model training finished ({time.time() - training_start_time:.2f}s).")
except Exception as e:
    print(f"   ERROR during final model training: {e}")
    raise

# --- 7. Prepare Input for Forecasting ---
print("\nPreparing input sequence for forecast...")
# Get the last n_timesteps of the scaled historical features
last_sequence_input_scaled = X_train_final_scaled[-n_timesteps:]
# Reshape for LSTM input (1 sample, n_timesteps, n_features)
forecast_input = last_sequence_input_scaled.reshape((1, n_timesteps, n_features))
print(f"Forecast input shape: {forecast_input.shape}")

# Get the last known actual level and date
last_known_level = y_train_level_final.iloc[-1].iloc[0] # Get scalar value
last_date = y_train_level_final.index[-1]
print(f"Last known actual level ({last_date.strftime('%Y-%m-%d')}): {last_known_level:.4f}")

# --- 8. Generate Forecast ---
print(f"Generating {num_forecast_steps}-step forecast...")
try:
    # Predict the vector of scaled percentage changes
    y_pred_scaled_vector = final_model.predict(forecast_input, verbose=0)[0]

    # Inverse scale the predictions
    y_pred_pct_change_vector = final_scaler_y.inverse_transform(y_pred_scaled_vector.reshape(1, -1)).flatten()

except Exception as e:
    print(f"   ERROR during forecast prediction: {e}")
    raise

# --- 9. Calculate Forecast Levels and Dates ---
forecast_results = []
current_pred_level = last_known_level

# Determine date frequency for generating future dates
inferred_freq = pd.infer_freq(X_train_final.index)
print(f"Inferred data frequency: {inferred_freq}")
if inferred_freq is None:
    print("Warning: Could not infer frequency. Assuming Quarterly Start ('QS-OCT' based on data). Adjust if needed.")
    # Assuming quarterly start if inference fails - adjust if needed
    # Common quarterly frequencies: 'QS', 'QS-JAN', 'QS-APR', 'QS-JUL', 'QS-OCT'
    # Check your last date (e.g., 2024-10-01) suggests QS-OCT or similar
    date_offset = pd.tseries.offsets.QuarterBegin(startingMonth=10) # Or QuarterEnd() etc.
else:
    # Use inferred frequency
    date_offset = pd.tseries.frequencies.to_offset(inferred_freq)
    print(f"Using offset: {date_offset}")

current_pred_date = last_date

for step in range(1, num_forecast_steps + 1):
    pred_pct_change = y_pred_pct_change_vector[step-1] # Index is step-1
    current_pred_level = current_pred_level * (1 + pred_pct_change)

    # Calculate the date for this forecast step
    current_pred_date = current_pred_date + date_offset
    forecast_date_str = current_pred_date.strftime('%Y-%m-%d')

    # Store results
    forecast_results.append({
        'Date': forecast_date_str,
        'LSTM_Forecast': current_pred_level
    })

# --- 10. Print Forecasts ---
forecast_df = pd.DataFrame(forecast_results)
forecast_df['Date'] = pd.to_datetime(forecast_df['Date'])
forecast_df = forecast_df.set_index('Date')

print(f"\n--- LSTM Forecasts for the next {num_forecast_steps} steps after {last_date.strftime('%Y-%m-%d')} ---")
print(forecast_df.round(4))

# --- 11. Visualization ---
print("\nGenerating final LSTM forecast plot...")
plt.figure(figsize=(15, 7))

# Plot historical data (all historical data used for training)
plt.plot(y_level_all.index, y_level_all[target_level_col].values, 'k-', label='Historical Actual Data', linewidth=1.5)

# Combine last historical point with forecasts for smooth line
lstm_plot_index = pd.Index([last_date]).union(forecast_df.index)
lstm_forecast_series = forecast_df['LSTM_Forecast'] if not forecast_df.empty else pd.Series(dtype=float)
lstm_plot_values = pd.concat([pd.Series([last_known_level], index=[last_date]), lstm_forecast_series])

# Plot forecast
best_units = best_hps.get('units')
best_lr = best_hps.get('learning_rate')
best_do = best_hps.get('dropout') if tune_dropout else 'NA'
forecast_label = f'LSTM Forecast (U={best_units}, LR={best_lr:.0e}, DO={best_do:.1f})'
plt.plot(lstm_plot_index, lstm_plot_values, 'r-p', label=forecast_label, linewidth=1.5, markersize=5) # Red pentagons

# Formatting
plt.title(f'House Index: Historical Data and {num_forecast_steps}-Step LSTM Forecast after {last_date.strftime("%Y-%m-%d")}', fontsize=14)
plt.ylabel('House Index', fontsize=12)
plt.xlabel('Date', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

# Set x-axis limits
display_history_years = 5
viz_start_date = last_date - pd.DateOffset(years=display_history_years)
viz_end_date = forecast_df.index.max() + pd.DateOffset(months=3) if not forecast_df.empty else last_date + pd.DateOffset(months=3) # Add padding
plt.xlim(viz_start_date, viz_end_date)

# Format date axis
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.gca().xaxis.set_minor_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45, ha='right')

plt.tight_layout()

# Save and Show Plot
try:
    save_fname = f'final_lstm_forecast_U{best_units}_LR{best_lr:.0e}_DO{best_do:.1f}.png'
    plt.savefig(save_fname, dpi=300, bbox_inches='tight')
    print(f"Plot saved as '{save_fname}'")
except Exception as e:
    print(f"Error saving plot: {e}")

plt.show()

print("\n--- Final LSTM forecasting process complete. ---")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
import time

# Ignore common warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

print("--- Starting Historical Mean Benchmark Walk-Forward Validation ---")

# --- Configuration ---
# Use the corrected input file
file_path = 'Preselect.csv'
# Target variable (percentage change)
target_pct_change_col = 'House_Index_pct_change'
# Original level column (for evaluation and iterative prediction)
target_level_col = 'House_Index'
# Date column name
date_col = 'Date'

# Define the training and testing periods
train_end_date_str = '2015-12-31' # End date for initial training period
test_start_date_str = '2016-01-01' # Start date for walk-forward validation
final_historical_date_str = '2024-10-01' # Last date of real historical data

# Walk-Forward Validation Setup
max_horizon = 8 # Maximum forecast steps ahead

# --- Helper Functions ---
def mean_absolute_percentage_error_np(y_true, y_pred):
    """Numpy implementation of MAPE"""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    if np.sum(mask) == 0: return np.nan
    if len(y_true[mask]) == 0: return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def directional_accuracy_np(y_true, y_pred, y_true_prev):
    """Numpy implementation of Directional Accuracy"""
    y_true, y_pred, y_true_prev = np.array(y_true), np.array(y_pred), np.array(y_true_prev)
    min_len = min(len(y_true), len(y_pred), len(y_true_prev))
    if min_len == 0: return np.nan
    y_true, y_pred, y_true_prev = y_true[:min_len], y_pred[:min_len], y_true_prev[:min_len]
    actual_diff = np.sign(y_true - y_true_prev)
    pred_diff = np.sign(y_pred - y_true_prev)
    correct_direction = (actual_diff == pred_diff).astype(int)
    return np.mean(correct_direction) * 100

# --- 1. Load Data ---
print(f"Loading data from '{file_path}'...")
try:
    df = pd.read_csv(file_path, index_col=date_col, parse_dates=True)
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    df.sort_index(inplace=True)
    print("Data loaded and indexed.")
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
    raise

# --- 2. Prepare Data ---
# Filter data to a specific end date if needed
final_historical_date = pd.to_datetime(final_historical_date_str)
df_filtered = df[df.index <= final_historical_date].copy()
print(f"Data filtered up to {final_historical_date_str}. Shape: {df_filtered.shape}")
if df_filtered.empty:
    raise ValueError("DataFrame is empty after filtering by final historical date.")

# Select necessary columns
required_cols = [target_pct_change_col, target_level_col]
missing_cols = [col for col in required_cols if col not in df_filtered.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df_selected = df_filtered[required_cols].copy()

# Handle Missing Values (Should be minimal if using corrected file)
initial_rows = len(df_selected)
df_selected.dropna(inplace=True)
rows_after_na_drop = len(df_selected)
if initial_rows > rows_after_na_drop:
    print(f"Dropped {initial_rows - rows_after_na_drop} rows containing NaNs.")
if df_selected.empty:
    raise ValueError("DataFrame empty after selecting columns and dropping NaNs.")

# Define target series
y_pct_change = df_selected[target_pct_change_col]
y_level = df_selected[target_level_col]

# --- 3. Split Data ---
train_end_date = pd.to_datetime(train_end_date_str)
test_start_date = pd.to_datetime(test_start_date_str)

# No explicit X_train needed, but define y_train for initial history
y_train_pct_change_initial = y_pct_change[y_pct_change.index <= train_end_date]
y_train_level_initial = y_level[y_level.index <= train_end_date]

# Define test set period
y_test_pct_change = y_pct_change[y_pct_change.index >= test_start_date]
y_test_level = y_level[y_level.index >= test_start_date] # Actual levels for evaluation

if y_test_level.empty or y_train_pct_change_initial.empty:
    raise ValueError("Training or Test set is empty after split.")

print(f"\nInitial training period ends: {train_end_date.strftime('%Y-%m-%d')}")
print(f"Test period: {y_test_level.index.min().strftime('%Y-%m-%d')} to {y_test_level.index.max().strftime('%Y-%m-%d')}")

# --- 4. Walk-Forward Validation ---

# Store predictions for each horizon
predictions_level = {h: [] for h in range(1, max_horizon + 1)}
actuals_level = {h: [] for h in range(1, max_horizon + 1)}
prev_actuals_level = {h: [] for h in range(1, max_horizon + 1)}
target_dates_level = {h: [] for h in range(1, max_horizon + 1)}

test_indices = y_test_level.index # Dates in the test period

start_time_wf = time.time()

# Initialize history
history_y_pct_change = y_train_pct_change_initial.copy()
history_y_level = y_train_level_initial.copy()

# Loop through potential forecast origins in the test set
# We need to stop early enough to have actuals for the max_horizon forecast
num_test_points = len(y_test_level)
num_origins = num_test_points - max_horizon + 1

if num_origins <= 0:
    raise ValueError(f"Test set too short ({num_test_points} points) for max_horizon={max_horizon}")

print(f"\nStarting walk-forward validation for Historical Mean ({num_origins} origins)...")

for i in range(num_origins):
    current_origin_date = test_indices[i] # The date *after* the history ends
    last_hist_date = test_indices[i-1] if i > 0 else train_end_date
    wf_step_start_time = time.time()

    print(f"Processing Origin {i + 1}/{num_origins} (History Ends: {last_hist_date.strftime('%Y-%m-%d')})...", end='\r')

    # --- 4.1. Calculate Expanding Historical Mean ---
    current_mean_pct_change = history_y_pct_change.mean()

    # --- 4.2. Generate Multi-Step Forecast ---
    last_known_actual_level = history_y_level.iloc[-1]
    current_pred_level = last_known_actual_level
    current_date = last_hist_date # Date corresponding to last_known_actual_level

    # Determine date frequency for generating future dates
    freq = pd.infer_freq(history_y_level.index)
    if freq is None:
        # Fallback assuming quarterly if inference fails
        offset = pd.tseries.offsets.QuarterBegin(startingMonth=last_hist_date.month) if hasattr(last_hist_date, 'month') else pd.DateOffset(months=3)
    else:
        offset = pd.tseries.frequencies.to_offset(freq)

    for h in range(1, max_horizon + 1):
        # Forecasted pct change is always the historical mean
        forecast_pct_change = current_mean_pct_change

        # Calculate predicted level
        current_pred_level = current_pred_level * (1 + forecast_pct_change)

        # Determine target date
        target_date = last_hist_date + (offset * h)

        # Store results if target date has actual data
        if target_date in y_test_level.index:
             predictions_level[h].append(current_pred_level)
             target_dates_level[h].append(target_date)
             actuals_level[h].append(y_test_level.loc[target_date])
             # Get previous actual for DA calculation
             prev_actual_date = target_date - offset
             if prev_actual_date in y_level.index: # Check in full y_level history
                 prev_actuals_level[h].append(y_level.loc[prev_actual_date])
             else: # Need to pop corresponding prediction/actual if prev is missing
                  predictions_level[h].pop()
                  target_dates_level[h].pop()
                  actuals_level[h].pop()
        # else: Don't store if actual doesn't exist for target_date

    # --- 4.3 Update History for the next outer loop iteration ---
    # Add the actual data point corresponding to the current origin date
    history_y_pct_change = pd.concat([history_y_pct_change, y_test_pct_change.iloc[[i]]])
    history_y_level = pd.concat([history_y_level, y_test_level.iloc[[i]]])

# End of outer loop
total_wf_time = time.time() - start_time_wf
print(f"\nWalk-forward validation complete. Total time: {total_wf_time:.2f} seconds.")


# --- 5. Calculate Metrics for Each Horizon ---
results_summary_mean = []
print("\n--- Calculating Historical Mean Benchmark Metrics ---")

for h in range(1, max_horizon + 1):
    print(f"\nHorizon: {h}-Step Ahead (Historical Mean)")
    preds = np.array(predictions_level[h])
    actuals = np.array(actuals_level[h])
    prev_actuals = np.array(prev_actuals_level[h])
    indices = pd.to_datetime(target_dates_level[h])

    # Ensure lengths match after potential removals due to missing actuals/prev
    min_len = min(len(preds), len(actuals), len(prev_actuals))
    preds = preds[:min_len]
    actuals = actuals[:min_len]
    prev_actuals = prev_actuals[:min_len]
    indices = indices[:min_len]

    if min_len > 0:
        mse = mean_squared_error(actuals, preds)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(actuals, preds)
        mape = mean_absolute_percentage_error_np(actuals, preds)
        da = directional_accuracy_np(actuals, preds, prev_actuals)

        print(f"  MSE: {mse:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}, MAPE: {mape:.2f}%, Dir. Acc.: {da:.2f}%")

        results_summary_mean.append({
            'Horizon': h, 'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'Dir_Acc': da,
        })
    else:
        print("   No valid predictions/actuals found for metric calculation.")
        results_summary_mean.append({'Horizon': h, 'MSE': np.nan, 'RMSE': np.nan, 'MAE': np.nan, 'MAPE': np.nan, 'Dir_Acc': np.nan})

# --- 6. Create Summary DataFrames ---
mean_results_df = pd.DataFrame(results_summary_mean)
mean_results_df.set_index('Horizon', inplace=True)

print("\n--- Detailed Historical Mean Benchmark Results by Forecast Horizon ---")
# Define column order including MSE
metric_cols_order = ['MSE', 'RMSE', 'MAE', 'MAPE', 'Dir_Acc']
existing_cols = [col for col in metric_cols_order if col in mean_results_df.columns]
print(mean_results_df[existing_cols].round({'MSE': 4, 'RMSE': 4, 'MAE': 4, 'MAPE': 2, 'Dir_Acc': 2}))


# --- Average Performance ---
avg_metrics_mean = mean_results_df.mean()
print(f"\n--- Average Historical Mean Benchmark Performance Across All Forecast Horizons (1-{max_horizon} Steps) ---")
print(f"  Avg MSE: {avg_metrics_mean['MSE']:.4f}")
print(f"  Avg RMSE: {avg_metrics_mean['RMSE']:.4f}")
print(f"  Avg MAE: {avg_metrics_mean['MAE']:.4f}")
print(f"  Avg MAPE (%): {avg_metrics_mean['MAPE']:.2f}")
print(f"  Avg Dir. Acc. (%): {avg_metrics_mean['Dir_Acc']:.2f}")

print("\nHistorical Mean Benchmark analysis complete.")

# Visualization
# You could add plotting code here similar to previous scripts,
# using y_train_level_initial, y_test_level, and predictions_level

# Code for visualization

# --- 9. Plotting Selected Horizons ---
# (Plotting code remains the same)
print("\n--- Plotting Results for Selected Horizons (h=1, 4, 8) ---")
plt.figure(figsize=(15, 7))
horizons_to_plot = [1, 4, 8]

# Create plot_dfs dictionary for plotting
plot_dfs = {}
for h in horizons_to_plot:
    if h in predictions_level and h in target_dates_level:
        plot_dfs[h] = pd.DataFrame({
            'Predicted_Level': predictions_level[h],
            'Actual_Level': actuals_level[h]
        }, index=target_dates_level[h])

# Determine the plotting range
min_plot_date = min((df.index.min() for h, df in plot_dfs.items() if not df.empty), default=train_end_date)
max_plot_date = max((df.index.max() for h, df in plot_dfs.items() if not df.empty), default=final_historical_date)
plot_start_hist_date = min_plot_date - pd.DateOffset(years=2)

# Subset actual data for plotting
actual_data_subset = y_level[(y_level.index >= plot_start_hist_date) & (y_level.index <= max_plot_date)]
plt.plot(actual_data_subset.index, actual_data_subset, label='Actual House Index', color='black', linewidth=1.5, alpha=0.7)

# Plot predictions for selected horizons
colors = plt.cm.viridis(np.linspace(0, 0.8, len(horizons_to_plot)))
markers = ['o', 's', '^']
for i, h in enumerate(horizons_to_plot):
    if h in plot_dfs and not plot_dfs[h].empty:
        results_df = plot_dfs[h]
        plt.plot(results_df.index, results_df['Predicted_Level'],
                 label=f'Predicted Level (h={h})', color=colors[i],
                 marker=markers[i % len(markers)], linestyle='--', markersize=4)

# Finalize the plot
plt.title('Multi-Step Forecasts vs Actual Level')
plt.xlabel('Date')
plt.ylabel('House Index')
plt.legend()
plt.grid(True, alpha=0.4)
plt.xlim(plot_start_hist_date, max_plot_date + pd.DateOffset(months=1))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
try:
    plt.savefig('forecast_comparison.png', dpi=300, bbox_inches='tight')
    print("Plot saved as 'forecast_comparison.png'")
except Exception as e:
    print(f"Error saving plot: {e}")
plt.show()

